In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
pip install mediapipe

In [ ]:
import os
import urllib.request
from pathlib import Path

model_dir = Path("/kaggle/working/models")
model_dir.mkdir(parents=True, exist_ok=True)

model_path = model_dir / "hand_landmarker.task"
url = "https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task"

if not model_path.exists():
    print("Downloading:", url)
    urllib.request.urlretrieve(url, model_path)
    print("Saved to:", model_path)
else:
    print("Already exists:", model_path)

print("Exists:", model_path.exists(), "Size(bytes):", model_path.stat().st_size if model_path.exists() else 0)


In [ ]:
"""
================================================================================
HANDWASH DATASET: DIRECTORY REMAPPING (12 → 7 CLASSES)
================================================================================
300 videos total: 25 per step × 12 steps
Merging left/right variants into single classes
================================================================================
"""

import shutil
from pathlib import Path
from tqdm import tqdm
import json
from collections import defaultdict

# ============================================================================
# CONFIGURATION - CORRECTED PATH!
# ============================================================================

INPUT_DIR = '/kaggle/input/datasets/realtimear/hand-wash-dataset/HandWashDataset/HandWashDataset'
OUTPUT_DIR = '/kaggle/working/handwash_7class_remapped'

# Directory name mapping: original → new class
DIRECTORY_MAPPING = {
    'Step_1': 1,
    'Step_2_Left': 2,
    'Step_2_Right': 2,     # MERGED with Step_2_Left
    'Step_3': 3,
    'Step_4_Left': 4,
    'Step_4_Right': 4,     # MERGED with Step_4_Left
    'Step_5_Left': 5,
    'Step_5_Right': 5,     # MERGED with Step_5_Left
    'Step_6_Left': 6,
    'Step_6_Right': 6,     # MERGED with Step_6_Left
    'Step_7_Left': 7,
    'Step_7_Right': 7,     # MERGED with Step_7_Left
}

print("="*80)
print("HANDWASH DATASET: DIRECTORY REMAPPING (12 → 7)")
print("="*80)
print(f"\n📂 Input: {INPUT_DIR}")
print(f"📂 Output: {OUTPUT_DIR}")
print(f"\n📊 Dataset: 300 videos total (25 per step)")

print(f"\n📋 Remapping plan:")
print("   Step_1 (25)                       → class_1 (25)")
print("   Step_2_Left (25) + Step_2_Right (25) → class_2 (50) MERGED")
print("   Step_3 (25)                       → class_3 (25)")
print("   Step_4_Left (25) + Step_4_Right (25) → class_4 (50) MERGED")
print("   Step_5_Left (25) + Step_5_Right (25) → class_5 (50) MERGED")
print("   Step_6_Left (25) + Step_6_Right (25) → class_6 (50) MERGED")
print("   Step_7_Left (25) + Step_7_Right (25) → class_7 (50) MERGED")

# Create output directories for 7 classes
for class_id in range(1, 8):
    Path(OUTPUT_DIR, f'class_{class_id}').mkdir(parents=True, exist_ok=True)

# ============================================================================
# SCAN INPUT DIRECTORIES
# ============================================================================

print(f"\n{'='*80}")
print("SCANNING INPUT DIRECTORIES")
print(f"{'='*80}\n")

input_path = Path(INPUT_DIR)
found_dirs = {}

for dir_name, target_class in DIRECTORY_MAPPING.items():
    dir_path = input_path / dir_name
    
    if dir_path.exists() and dir_path.is_dir():
        # Count video files
        video_files = list(dir_path.glob('*.mp4')) + \
                     list(dir_path.glob('*.avi')) + \
                     list(dir_path.glob('*.mov')) + \
                     list(dir_path.glob('*.MP4')) + \
                     list(dir_path.glob('*.AVI')) + \
                     list(dir_path.glob('*.MOV'))
        
        found_dirs[dir_name] = {
            'path': dir_path,
            'target_class': target_class,
            'file_count': len(video_files),
            'files': video_files
        }
        
        print(f"OK {dir_name:20s} -> class_{target_class}  ({len(video_files):3d} videos)")
    else:
        print(f"XX {dir_name:20s} -> NOT FOUND")

# ============================================================================
# COPY FILES TO NEW STRUCTURE
# ============================================================================

print(f"\n{'='*80}")
print("COPYING FILES TO NEW STRUCTURE")
print(f"{'='*80}\n")

stats = {
    'total_files': 0,
    'files_per_class': defaultdict(int),
    'videos_per_original_dir': {},
}

print("Processing:\n")

for dir_name, info in sorted(found_dirs.items()):
    target_class = info['target_class']
    files = info['files']
    
    print(f"   {dir_name:20s} -> class_{target_class}", end="  ")
    
    # Copy each file
    for file_path in files:
        # Create new filename to preserve origin and avoid conflicts
        new_filename = f"{dir_name}_{file_path.name}"
        dest_path = Path(OUTPUT_DIR) / f'class_{target_class}' / new_filename
        
        # Copy file
        shutil.copy2(file_path, dest_path)
        
        stats['total_files'] += 1
        stats['files_per_class'][target_class] += 1
    
    stats['videos_per_original_dir'][dir_name] = len(files)
    print(f"OK Copied {len(files)} videos")

# ============================================================================
# FINAL STATISTICS
# ============================================================================

print(f"\n{'='*80}")
print("REMAPPING COMPLETE!")
print(f"{'='*80}\n")

print(f"Summary:")
print(f"   Total videos processed: {stats['total_files']:,}\n")

print(f"Original distribution (12 directories):")
for dir_name in sorted(stats['videos_per_original_dir'].keys()):
    count = stats['videos_per_original_dir'][dir_name]
    target_class = DIRECTORY_MAPPING[dir_name]
    print(f"   {dir_name:20s} -> class_{target_class}  |  {count:3d} videos")

print(f"\nNew distribution (7 classes - MERGED):")
total_files = stats['total_files']
for class_id in range(1, 8):
    count = stats['files_per_class'][class_id]
    percentage = (count / total_files * 100) if total_files > 0 else 0
    
    # Indicate which classes are merged
    if class_id in [2, 4, 5, 6, 7]:
        merged_tag = "MERGED (left + right)"
    else:
        merged_tag = ""
    
    print(f"   class_{class_id}: {count:4d} videos ({percentage:5.2f}%)  {merged_tag}")

# Check for class imbalance
counts = [stats['files_per_class'][i] for i in range(1, 8) if stats['files_per_class'][i] > 0]
if counts:
    max_count = max(counts)
    min_count = min(counts)
    imbalance_ratio = max_count / min_count if min_count > 0 else 0
    
    print(f"\nClass balance:")
    print(f"   Largest class: {max_count} videos")
    print(f"   Smallest class: {min_count} videos")
    print(f"   Imbalance ratio: {imbalance_ratio:.2f}:1")
    
    if imbalance_ratio > 2.0:
        print(f"   WARNING: Moderate imbalance - Consider data augmentation!")
    else:
        print(f"   OK: Well balanced dataset!")

# ============================================================================
# SAVE METADATA
# ============================================================================

metadata = {
    'input_directory': str(INPUT_DIR),
    'output_directory': str(OUTPUT_DIR),
    'total_videos': stats['total_files'],
    'original_classes': 12,
    'remapped_classes': 7,
    'directory_mapping': DIRECTORY_MAPPING,
    'videos_per_class': dict(stats['files_per_class']),
    'videos_per_original_dir': stats['videos_per_original_dir'],
    'class_descriptions': {
        1: 'Step 1',
        2: 'Step 2 (merged: left + right)',
        3: 'Step 3',
        4: 'Step 4 (merged: left + right)',
        5: 'Step 5 (merged: left + right)',
        6: 'Step 6 (merged: left + right)',
        7: 'Step 7 (merged: left + right)',
    }
}

metadata_path = Path(OUTPUT_DIR) / 'remapping_metadata.json'
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"\nMetadata saved: {metadata_path}")

# ============================================================================
# VERIFY OUTPUT STRUCTURE
# ============================================================================

print(f"\n{'='*80}")
print("OUTPUT DIRECTORY STRUCTURE")
print(f"{'='*80}\n")

print(f"{OUTPUT_DIR}/")
for class_id in range(1, 8):
    class_dir = Path(OUTPUT_DIR) / f'class_{class_id}'
    video_count = len(list(class_dir.glob('*.mp4')))
    
    # Show sample filenames
    sample_files = list(class_dir.glob('*.mp4'))[:2]
    
    print(f"   |-- class_{class_id}/  ({video_count} videos)")
    for sample in sample_files:
        print(f"   |   |-- {sample.name}")
    if video_count > 2:
        print(f"   |   +-- ... ({video_count - 2} more)")

print(f"\n{'='*80}")
print("DATASET READY!")
print(f"{'='*80}\n")

print("Next steps:")
print("1. OK Class remapping complete (12 -> 7)")
print("2. OK Dataset is balanced (except class_1 and class_3 have 25 vs 50)")
print("3. OK Ready for frame extraction or direct video processing")
print("4. OK Ready for CNN+LSTM model training")

print(f"\nUse this path for your CNN+LSTM training:")
print(f"   DATA_DIR = '{OUTPUT_DIR}'")

print(f"\n{'='*80}\n")

In [ ]:
%%writefile config.py
"""
config.py — centralised configuration for the HandWash TCN pipeline
=====================================================================
v5 changes:
  - FEATURE_DIM updated 399 → 403  (+4 Step 6 v5 disambiguation dims)
    New dims: curl angle × 2 (399:401) + spread velocity × 2 (401:403)
  - Added STEP6_V5_DIM, STEP6_V5_START, STEP6_V5_END constants
  - PROTECT_START unchanged at 384 — already covers 384:403 by range
  - LR_DUALSTREAM added for dual-stream fast training
  - DROPOUT_DUALSTREAM added
  - All other settings carried forward from v4
"""

import sys
import os

_WORKING_DIR = "/kaggle/working"
if _WORKING_DIR not in sys.path:
    sys.path.insert(0, _WORKING_DIR)

from pathlib import Path

# ============================================================================
# DATA PATHS
# ============================================================================

RAW_DATA_DIR = Path("/kaggle/working/handwash_7class_remapped")
OUT_DIR      = Path("/kaggle/working/handwash_tcn_dataset")
SEQ_DIR      = OUT_DIR / "sequences"
META_DIR     = OUT_DIR / "metadata"
CKPT_DIR     = OUT_DIR / "checkpoints"

# ============================================================================
# FEATURE DIMENSIONS  (v5)
# ============================================================================

KINEMATIC_DIM   = 384    # 3 × 128  (wrist-relative + velocity + acceleration)
ORIENT_DIM      = 9      # palm normals (6) + dot product (1) + z-spread (2)
STEP6_V4_DIM    = 6      # fingertip spread (4) + fingertip-to-wrist (2)
STEP6_V5_DIM    = 4      # curl angle (2) + spread velocity (2)  [NEW in v5]
FEATURE_DIM     = KINEMATIC_DIM + ORIENT_DIM + STEP6_V4_DIM + STEP6_V5_DIM  # 403

# Slice boundaries within the 403-dim vector
ORIENT_SLICE_START = 384
ORIENT_SLICE_END   = 393
STEP6_V4_START     = 393
STEP6_V4_END       = 399
STEP6_V5_START     = 399    # NEW
STEP6_V5_END       = 403    # NEW

# All engineered dims protected from feature dropout (covers 384:403)
PROTECT_START = 384

# ============================================================================
# SEQUENCE PARAMETERS
# ============================================================================

SEQ_LEN    = 64
STRIDE     = 16
SEED       = 42
TARGET_FPS = 15

# ============================================================================
# SUBJECT CONFIGURATION & CROSS-VALIDATION
# ============================================================================

ALL_SUBJECTS = ["G01", "G02", "G03", "G04", "G05"]

LOSO_TEST_SUBJECT = None   # set to "G01".."G05" for LOSO-CV

FIXED_VAL_SUBJECT  = "G03"
FIXED_TEST_SUBJECT = "G04"

NUM_CLASSES = 7

# ============================================================================
# HAND MIRRORING AUGMENTATION
# ============================================================================

MIRROR_LABELS = {0, 2}   # Steps 1 and 3 — minority classes

# ============================================================================
# MEDIAPIPE CONFIGURATION
# ============================================================================

MP_TASK_MODEL = "/kaggle/working/models/hand_landmarker.task"
MAX_NUM_HANDS = 2
MP_DET_CONF   = 0.5
MP_TRK_CONF   = 0.5
MP_PRES_CONF  = 0.5

# ============================================================================
# TRAINING PARAMETERS
# ============================================================================

BATCH_SIZE  = 64
NUM_WORKERS = 2

EPOCHS_TEACHER = 150
EPOCHS_STUDENT = 150

LR_TEACHER        = 1e-3
LR_STUDENT        = 8e-4
LR_DUALSTREAM     = 5e-4
WEIGHT_DECAY      = 1e-4

# ============================================================================
# KNOWLEDGE DISTILLATION PARAMETERS
# ============================================================================

KD_T     = 3.0
KD_ALPHA = 0.6
KD_BETA  = 0.35
KD_GAMMA = 0.35

# ============================================================================
# CLASS WEIGHTS  (sqrt-inverse-frequency — used as alpha in focal loss)
# ============================================================================

CLASS_WEIGHTS = [
    1.414,  # Step 1  — minority
    1.000,  # Step 2
    1.414,  # Step 3  — minority
    1.000,  # Step 4
    1.000,  # Step 5
    1.000,  # Step 6
    1.000,  # Step 7
]

# ============================================================================
# DATA AUGMENTATION PARAMETERS
# ============================================================================

USE_AUGMENTATION   = True
MIXUP_ALPHA        = 0.4
TEMPORAL_SHIFT_MAX = 8
LANDMARK_NOISE_STD = 0.01
HAND_DROP_PROB     = 0.1

# Protect all engineered dims (384:403) from feature dropout.
# Covers: orientation (384:393), step6-v4 (393:399), step6-v5 (399:403)
AUG_ORIENT_PROTECT = True

# ============================================================================
# REGULARIZATION
# ============================================================================

LABEL_SMOOTHING     = 0.1
DROPOUT_TEACHER     = 0.3
DROPOUT_STUDENT     = 0.25
DROPOUT_DUALSTREAM  = 0.3
EARLY_STOP_PATIENCE = 25
GRAD_CLIP           = 1.0

# ============================================================================
# MODEL ARCHITECTURE
# ============================================================================

# Teacher model
TEACHER_BRANCH_KERNELS  = (3, 5, 7)
TEACHER_BRANCH_CHANNELS = 128
TEACHER_DILATIONS       = (1, 2, 4, 8)
TEACHER_FUSION_DIM      = 192
TEACHER_GRU_HIDDEN      = 128

# Student model (~63% of teacher capacity)
STUDENT_BRANCH_KERNELS  = (3, 5, 7)
STUDENT_BRANCH_CHANNELS = 96
STUDENT_DILATIONS       = (1, 2, 4, 8)
STUDENT_FUSION_DIM      = 144
STUDENT_GRU_HIDDEN      = 96

In [ ]:
%%writefile build_sequences_v6.py
"""
build_sequences_v6.py — Feature extraction pipeline v6
=======================================================
Changes from v5:

1. PALM-SCALE NORMALIZATION for distance-like Step 6 features
   Problem: raw MediaPipe coordinate distances (fingertip spread,
   fingertip-to-wrist, z-spread) vary with hand size, camera distance,
   and camera angle. Per-feature z-score normalization removes global
   mean/variance but cannot remove within-subject correlation between
   hand size and feature value. This is the most likely explanation
   for G05 LOSO collapse — G05's hand geometry sits outside the
   training distribution.

   Fix: before computing any distance-like feature, divide raw
   landmark coordinates by a palm scale factor:
       scale = ||landmarks[MIDDLE_MCP] - landmarks[WRIST]||
   This makes all distance features invariant to hand size and
   camera distance. Angular features (curl angle, palm normal dot
   product) and presence flags are left unscaled — they are already
   scale-invariant.

   Affected feature dims (all distance-based):
     [0:126]   wrist-relative coords  ← scaled before wrist-subtraction
     [126:128] presence flags         ← unscaled
     [384:393] orientation: palm normals (unit vectors, scale-invariant),
               dot product (scale-invariant), z-spread (scaled)
     [393:399] step6-v4: spread mean/std, fingertip-to-wrist (scaled)
     [399:401] step6-v5: curl angles (scale-invariant, angle-based)
     [401:403] step6-v5: spread velocity (scaled, derived from spread)

   Implementation: _palm_scale(landmarks, present) returns the
   wrist-to-middle-MCP distance, or 1.0 if hand absent (no-op).
   Applied in build_full_feature_sequence_v6() before all feature
   computation on raw coords.

2. Step 6 added to MIRROR_LABELS (was {0, 2}, now {0, 2, 5})
   Step 6 involves fingertips pressing into the opposite palm.
   Left-hand-dominant and right-hand-dominant versions are
   geometrically distinct. Mirroring adds free augmentation
   diversity for the weakest class.

3. OUT_DIR changed to handwash_tcn_dataset_v6 to keep v5 intact
   so you can compare LOSO results between versions.

All other pipeline logic (kinematic block, orientation, step6-v4/v5
features, normalization, sequence windowing) is identical to v5.
FEATURE_DIM remains 403.
"""

import sys
_WORKING_DIR = "/kaggle/working"
if _WORKING_DIR not in sys.path:
    sys.path.insert(0, _WORKING_DIR)

import json
import re
import random
from pathlib import Path

import numpy as np
import cv2
import mediapipe as mp
from tqdm import tqdm

from config import (
    RAW_DATA_DIR, SEQ_LEN, STRIDE, SEED, TARGET_FPS,
    MP_TASK_MODEL, MAX_NUM_HANDS, MP_DET_CONF, MP_TRK_CONF, MP_PRES_CONF,
    ALL_SUBJECTS, LOSO_TEST_SUBJECT, FIXED_VAL_SUBJECT, FIXED_TEST_SUBJECT,
)

# ─────────────────────────────────────────────────────────────────────────────
# V6 OVERRIDES
# ─────────────────────────────────────────────────────────────────────────────

# Keep v5 output intact — write v6 to a separate directory
OUT_DIR_V6 = Path("/kaggle/working/handwash_tcn_dataset_v6")
SEQ_DIR_V6 = OUT_DIR_V6 / "sequences"
META_DIR_V6 = OUT_DIR_V6 / "metadata"
CKPT_DIR_V6 = OUT_DIR_V6 / "checkpoints"

# Step 6 now mirrored alongside minority classes
MIRROR_LABELS_V6 = {0, 2, 5}   # Steps 1, 3, 6

# ─────────────────────────────────────────────────────────────────────────────
# CONSTANTS  (unchanged from v5)
# ─────────────────────────────────────────────────────────────────────────────

VIDEO_EXTS    = {".mp4", ".avi", ".mov", ".MP4", ".AVI", ".MOV"}
HAND_COORDS   = 63
BASE_DIM      = 128
_FLAG_OFFSETS = [126, 127]

# MediaPipe landmark indices
WRIST      = 0
MIDDLE_MCP = 9     # used for palm scale
INDEX_MCP  = 5
PINKY_MCP  = 17
INDEX_TIP  = 8
MIDDLE_TIP = 12
RING_TIP   = 16
PINKY_TIP  = 20
THUMB_TIP  = 4

KINEMATIC_DIM  = 384
ORIENT_DIM     = 9
STEP6_V4_DIM   = 6
STEP6_V5_DIM   = 4
FEATURE_DIM    = KINEMATIC_DIM + ORIENT_DIM + STEP6_V4_DIM + STEP6_V5_DIM  # 403

ORIENT_START   = 384
ORIENT_END     = 393
STEP6_V4_START = 393
STEP6_V4_END   = 399
STEP6_V5_START = 399
STEP6_V5_END   = 403
PROTECT_START  = ORIENT_START  # 384


# ─────────────────────────────────────────────────────────────────────────────
# UTILITIES
# ─────────────────────────────────────────────────────────────────────────────

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)


def _flag_indices():
    return [b * 128 + f for b in range(3) for f in _FLAG_OFFSETS]


def extract_subject_id(path_or_name):
    m = re.search(r"G_?(\d+)", str(path_or_name), re.IGNORECASE)
    if m:
        return f"G{int(m.group(1)):02d}"
    raise ValueError(f"Could not extract subject ID from: {path_or_name}")


# ─────────────────────────────────────────────────────────────────────────────
# VIDEO LISTING  (same as v5)
# ─────────────────────────────────────────────────────────────────────────────

def list_videos():
    items = []
    for class_id in range(1, 8):
        class_dir = RAW_DATA_DIR / f"class_{class_id}"
        if not class_dir.exists():
            continue
        for p in sorted(class_dir.iterdir()):
            if p.suffix in VIDEO_EXTS:
                items.append({
                    "video_path": str(p),
                    "video_name": p.stem,
                    "label":      class_id - 1,
                    "subject":    extract_subject_id(p),
                })
    return items


def split_videos(video_items, loso_test_subject,
                 fixed_val_subject, fixed_test_subject):
    if loso_test_subject is not None:
        test_subj = loso_test_subject
        val_subj  = next(s for s in ALL_SUBJECTS if s != test_subj)
        train_subj = [s for s in ALL_SUBJECTS
                      if s not in (test_subj, val_subj)]
        print(f"\nLOSO fold  test={test_subj}  val={val_subj}  train={train_subj}")
    else:
        test_subj  = fixed_test_subject
        val_subj   = fixed_val_subject
        train_subj = [s for s in ALL_SUBJECTS
                      if s not in (test_subj, val_subj)]
        print(f"\nFixed split  test={test_subj}  val={val_subj}  train={train_subj}")

    def _assign(s):
        if s == test_subj: return "test"
        if s == val_subj:  return "val"
        return "train"

    split      = {i: _assign(v["subject"]) for i, v in enumerate(video_items)}
    labels_arr = np.array([v["label"]   for v in video_items])
    groups_arr = np.array([v["subject"] for v in video_items])

    for sname in ("train", "val", "test"):
        idxs   = [i for i, s in split.items() if s == sname]
        subjs  = sorted(set(groups_arr[idxs]))
        counts = np.bincount(labels_arr[idxs], minlength=7)
        print(f"  {sname:5s}: {len(idxs):4d} videos | "
              f"subjects {subjs} | class counts {counts.tolist()}")
    return split


# ─────────────────────────────────────────────────────────────────────────────
# MEDIAPIPE EXTRACTION  (same as v5)
# ─────────────────────────────────────────────────────────────────────────────

def result_to_feature(result):
    left  = np.zeros((21, 3), dtype=np.float32)
    right = np.zeros((21, 3), dtype=np.float32)
    left_present, right_present = 0.0, 0.0

    if result.hand_landmarks:
        for i, lms in enumerate(result.hand_landmarks):
            hand_label = ""
            if i < len(result.handedness) and len(result.handedness[i]) > 0:
                hand_label = result.handedness[i][0].category_name
            coords = np.array([[lm.x, lm.y, lm.z] for lm in lms],
                               dtype=np.float32)
            if hand_label == "Left":
                left, left_present = coords, 1.0
            else:
                right, right_present = coords, 1.0

    return np.concatenate([
        left.reshape(-1), right.reshape(-1),
        np.array([left_present, right_present], dtype=np.float32),
    ])


def extract_base_features(video_path, target_fps=15):
    if not Path(MP_TASK_MODEL).exists():
        raise FileNotFoundError(f"HandLandmarker not found: {MP_TASK_MODEL}")

    BaseOptions           = mp.tasks.BaseOptions
    HandLandmarker        = mp.tasks.vision.HandLandmarker
    HandLandmarkerOptions = mp.tasks.vision.HandLandmarkerOptions
    VisionRunningMode     = mp.tasks.vision.RunningMode

    options = HandLandmarkerOptions(
        base_options=BaseOptions(model_asset_path=MP_TASK_MODEL),
        running_mode=VisionRunningMode.VIDEO,
        num_hands=MAX_NUM_HANDS,
        min_hand_detection_confidence=MP_DET_CONF,
        min_hand_presence_confidence=MP_PRES_CONF,
        min_tracking_confidence=MP_TRK_CONF,
    )

    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        return None

    original_fps   = cap.get(cv2.CAP_PROP_FPS) or 30.0
    frame_interval = max(1, int(round(original_fps / target_fps)))
    feats, frame_idx, sampled_idx = [], 0, 0

    with HandLandmarker.create_from_options(options) as landmarker:
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            if frame_idx % frame_interval == 0:
                rgb      = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
                ts_ms    = int((sampled_idx / target_fps) * 1000.0)
                result   = landmarker.detect_for_video(mp_image, ts_ms)
                feats.append(result_to_feature(result))
                sampled_idx += 1
            frame_idx += 1

    cap.release()
    if len(feats) < SEQ_LEN:
        return None
    return np.stack(feats, axis=0)


# ─────────────────────────────────────────────────────────────────────────────
# HAND MIRRORING  (same as v5)
# ─────────────────────────────────────────────────────────────────────────────

def mirror_raw_frame(frame):
    m = frame.copy()
    for lm in range(21):
        m[lm * 3]               = -frame[lm * 3]
        m[HAND_COORDS + lm * 3] = -frame[HAND_COORDS + lm * 3]
    left_block, right_block = m[0:63].copy(), m[63:126].copy()
    m[0:63], m[63:126] = right_block, left_block
    m[126], m[127] = m[127], m[126]
    return m


def mirror_base_sequence(base_seq):
    return np.stack(
        [mirror_raw_frame(base_seq[t]) for t in range(base_seq.shape[0])],
        axis=0,
    )


# ─────────────────────────────────────────────────────────────────────────────
# PALM SCALE  — NEW IN V6
# ─────────────────────────────────────────────────────────────────────────────

def _palm_scale(landmarks: np.ndarray, present: float) -> float:
    """
    Returns the Euclidean distance from WRIST to MIDDLE_MCP in raw
    MediaPipe coordinates. Used to normalize all distance-like features
    so they become invariant to hand size and camera distance.

    Returns 1.0 (no-op) when the hand is absent, preserving zero-padding.

    Why WRIST → MIDDLE_MCP:
      This segment is the most stable across all hand poses — it does not
      change length regardless of finger configuration, unlike fingertip
      distances which shorten when fingers curl. It captures hand size
      and camera distance reliably.

    Typical values in MediaPipe normalized coords:
      ~0.05–0.15 depending on camera distance and hand size.
    """
    if present < 0.5:
        return 1.0
    scale = float(np.linalg.norm(landmarks[MIDDLE_MCP] - landmarks[WRIST]))
    return scale if scale > 1e-6 else 1.0


def _scale_landmarks(landmarks: np.ndarray,
                     present: float) -> tuple:
    """
    Returns (scaled_landmarks, scale_factor).
    scaled_landmarks has the same shape as landmarks but all coordinates
    divided by the palm scale. When hand is absent, returns landmarks
    unchanged and scale=1.0.
    """
    s = _palm_scale(landmarks, present)
    return landmarks / s, s


# ─────────────────────────────────────────────────────────────────────────────
# HAND LANDMARK ACCESSORS
# ─────────────────────────────────────────────────────────────────────────────

def _get_hand_landmarks(frame, hand):
    if hand == "left":
        return frame[0:63].reshape(21, 3), float(frame[126])
    return frame[63:126].reshape(21, 3), float(frame[127])


def _palm_normal(landmarks):
    v1     = landmarks[INDEX_MCP] - landmarks[WRIST]
    v2     = landmarks[PINKY_MCP] - landmarks[WRIST]
    normal = np.cross(v1, v2).astype(np.float32)
    norm   = np.linalg.norm(normal)
    return normal / norm if norm > 1e-6 else normal


# ─────────────────────────────────────────────────────────────────────────────
# ORIENTATION FEATURES  (9 dims — angles are scale-invariant, z-spread scaled)
# ─────────────────────────────────────────────────────────────────────────────

def _fingertip_z_spread(landmarks, present):
    if present < 0.5:
        return 0.0
    tips_z = landmarks[[INDEX_TIP, MIDDLE_TIP, RING_TIP, PINKY_TIP], 2]
    return float(tips_z.max() - tips_z.min())


def compute_orientation_features_v6(raw_frame):
    """
    Same as v5 but uses palm-scaled landmarks for z-spread.
    Palm normals and dot product are computed on unit vectors — inherently
    scale-invariant — so they do not need scaling.
    """
    left_lm,  lp = _get_hand_landmarks(raw_frame, "left")
    right_lm, rp = _get_hand_landmarks(raw_frame, "right")

    # Scale landmarks for distance-based features
    left_scaled,  _ = _scale_landmarks(left_lm,  lp)
    right_scaled, _ = _scale_landmarks(right_lm, rp)

    # Palm normals: computed on raw (unit vectors anyway)
    ln = _palm_normal(left_lm)  if lp > 0.5 else np.zeros(3, np.float32)
    rn = _palm_normal(right_lm) if rp > 0.5 else np.zeros(3, np.float32)

    return np.array([
        *ln, *rn,
        float(np.dot(ln, rn)),
        _fingertip_z_spread(left_scaled,  lp),   # scaled
        _fingertip_z_spread(right_scaled, rp),   # scaled
    ], dtype=np.float32)


# ─────────────────────────────────────────────────────────────────────────────
# STEP 6 V4 FEATURES  (6 dims — all distance-based, now palm-scaled)
# ─────────────────────────────────────────────────────────────────────────────

def _inter_fingertip_spread(landmarks, present):
    if present < 0.5:
        return 0.0, 0.0
    tips = landmarks[[INDEX_TIP, MIDDLE_TIP, RING_TIP, PINKY_TIP]]
    dists = np.array([
        np.linalg.norm(tips[0] - tips[1]),
        np.linalg.norm(tips[1] - tips[2]),
        np.linalg.norm(tips[2] - tips[3]),
    ], dtype=np.float32)
    return float(dists.mean()), float(dists.std())


def _fingertip_to_opposite_wrist(own_lm, own_p, opp_lm, opp_p):
    if own_p < 0.5 or opp_p < 0.5:
        return 0.0
    tips  = own_lm[[INDEX_TIP, MIDDLE_TIP, RING_TIP, PINKY_TIP]]
    dists = np.linalg.norm(tips - opp_lm[WRIST], axis=1)
    return float(dists.mean())


def compute_step6_v4_features_v6(raw_frame):
    """
    Same as v5 but ALL distance features are computed on palm-scaled
    landmarks. This makes fingertip spread and fingertip-to-wrist
    distances invariant to hand size and camera distance.
    """
    left_lm,  lp = _get_hand_landmarks(raw_frame, "left")
    right_lm, rp = _get_hand_landmarks(raw_frame, "right")

    # Scale each hand independently by its own palm scale
    left_scaled,  _ = _scale_landmarks(left_lm,  lp)
    right_scaled, _ = _scale_landmarks(right_lm, rp)

    l_mean, l_std = _inter_fingertip_spread(left_scaled,  lp)
    r_mean, r_std = _inter_fingertip_spread(right_scaled, rp)

    # Cross-hand distance: use each hand scaled by its OWN palm
    # (opposite wrist is from the other hand's raw coords divided by
    # that hand's scale — both hands normalized independently)
    l_to_r = _fingertip_to_opposite_wrist(
        left_scaled, lp, right_scaled, rp)
    r_to_l = _fingertip_to_opposite_wrist(
        right_scaled, rp, left_scaled, lp)

    return np.array([l_mean, l_std, r_mean, r_std, l_to_r, r_to_l],
                    dtype=np.float32)


# ─────────────────────────────────────────────────────────────────────────────
# STEP 6 V5 FEATURES  (4 dims — curl angles scale-invariant, spread vel scaled)
# ─────────────────────────────────────────────────────────────────────────────

def _fingertip_curl_angle(landmarks, present):
    """
    Angle between wrist-to-fingertip-centroid vector and palm normal.
    This is computed from unit vectors so it is scale-invariant.
    No change needed from v5.
    """
    if present < 0.5:
        return 0.0
    tip_indices  = [INDEX_TIP, MIDDLE_TIP, RING_TIP, PINKY_TIP]
    tip_centroid = landmarks[tip_indices].mean(axis=0)
    w2t          = tip_centroid - landmarks[WRIST]
    norm_w2t     = np.linalg.norm(w2t)
    if norm_w2t < 1e-6:
        return 0.0
    w2t    = w2t / norm_w2t
    palm_n = _palm_normal(landmarks)
    norm_pn = np.linalg.norm(palm_n)
    if norm_pn < 1e-6:
        return 0.0
    cos_angle = float(np.clip(np.dot(w2t, palm_n), -1.0, 1.0))
    return float(np.arccos(cos_angle))


def compute_step6_v5_per_frame_v6(raw_frame):
    """Curl angles — scale-invariant, same as v5."""
    left_lm,  lp = _get_hand_landmarks(raw_frame, "left")
    right_lm, rp = _get_hand_landmarks(raw_frame, "right")
    return np.array([
        _fingertip_curl_angle(left_lm,  lp),
        _fingertip_curl_angle(right_lm, rp),
    ], dtype=np.float32)


def compute_spread_velocity(step6_v4_seq):
    """
    Temporal diff of spread mean — same as v5.
    Since spread mean is now palm-scaled, velocity is also scale-invariant.
    """
    left_spread_mean  = step6_v4_seq[:, 0]
    right_spread_mean = step6_v4_seq[:, 2]
    left_vel  = np.diff(left_spread_mean,  prepend=left_spread_mean[0])
    right_vel = np.diff(right_spread_mean, prepend=right_spread_mean[0])
    return np.stack([left_vel, right_vel], axis=1).astype(np.float32)


# ─────────────────────────────────────────────────────────────────────────────
# KINEMATIC BLOCK  — v6: wrist-relative computed on palm-scaled coords
# ─────────────────────────────────────────────────────────────────────────────

def _apply_palm_scale_to_base_frame(frame: np.ndarray) -> np.ndarray:
    """
    Divide each hand's 21 landmarks by that hand's palm scale factor.
    Presence flags (dims 126, 127) are left unchanged.

    This operates on the raw 128-dim base frame before any kinematic
    feature engineering.
    """
    out = frame.copy()

    left_lm  = frame[0:63].reshape(21, 3)
    right_lm = frame[63:126].reshape(21, 3)
    lp       = float(frame[126])
    rp       = float(frame[127])

    sl = _palm_scale(left_lm,  lp)
    sr = _palm_scale(right_lm, rp)

    out[0:63]   = (left_lm  / sl).reshape(-1)
    out[63:126] = (right_lm / sr).reshape(-1)
    # flags 126, 127 unchanged

    return out


def _apply_palm_scale_to_base_sequence(base_seq: np.ndarray) -> np.ndarray:
    """Apply palm scale to every frame in a (T, 128) base sequence."""
    return np.stack(
        [_apply_palm_scale_to_base_frame(base_seq[t])
         for t in range(base_seq.shape[0])],
        axis=0,
    )


def _make_wrist_relative(base_seq):
    seq         = base_seq.copy()
    left_wrist  = seq[:, 0:3].copy()
    right_wrist = seq[:, HAND_COORDS: HAND_COORDS + 3].copy()
    for lm in range(21):
        seq[:, lm * 3: lm * 3 + 3]                              -= left_wrist
        seq[:, HAND_COORDS + lm * 3: HAND_COORDS + lm * 3 + 3] -= right_wrist
    return seq


# ─────────────────────────────────────────────────────────────────────────────
# FULL FEATURE SEQUENCE  — v6 main change
# ─────────────────────────────────────────────────────────────────────────────

def build_full_feature_sequence_v6(base_seq: np.ndarray) -> np.ndarray:
    """
    Full feature engineering pipeline — v6.

    Changes from v5:
      1. Palm-scale normalization applied to raw base_seq BEFORE all
         feature computation. This makes all distance-based features
         (kinematic coords, spread, fingertip-to-wrist) scale-invariant.
      2. Step 6 v4/v5 features computed on palm-scaled landmarks.
      3. Orientation z-spread computed on palm-scaled landmarks.
      4. Curl angles and dot products are unchanged (inherently scale-inv).

    Pipeline:
      raw (T, 128)
        → palm-scale each frame (÷ wrist-to-middle-MCP distance per hand)
        → wrist-relative coords
        → velocity (Δ wrist-relative)
        → acceleration (Δ velocity)
        → orientation features (9 dims)     from palm-scaled raw coords
        → step6 v4 features    (6 dims)     from palm-scaled raw coords
        → step6 v5 curl angles (2 dims)     from raw coords (scale-inv)
        → step6 v5 spread vel  (2 dims)     from palm-scaled step6_v4 seq
        → concatenate → (T, 403)

    FEATURE_DIM = 403 (unchanged from v5).
    """
    T = base_seq.shape[0]

    # ── Apply palm scale to raw coordinates ──────────────────────────────
    scaled_seq = _apply_palm_scale_to_base_sequence(base_seq)

    # ── Kinematic block (384 dims) from palm-scaled coords ───────────────
    wr_seq    = _make_wrist_relative(scaled_seq)
    vel       = np.diff(wr_seq, axis=0, prepend=wr_seq[0:1])
    acc       = np.diff(vel,    axis=0, prepend=vel[0:1])
    kinematic = np.concatenate([wr_seq, vel, acc], axis=1)     # (T, 384)

    # ── Orientation block (9 dims) — v6 uses palm-scaled raw ─────────────
    orient = np.stack(
        [compute_orientation_features_v6(scaled_seq[t]) for t in range(T)],
        axis=0,
    )   # (T, 9)

    # ── Step6 v4 block (6 dims) — v6 uses palm-scaled raw ────────────────
    step6_v4 = np.stack(
        [compute_step6_v4_features_v6(scaled_seq[t]) for t in range(T)],
        axis=0,
    )   # (T, 6)

    # ── Step6 v5 curl angle block (2 dims) — scale-invariant ─────────────
    curl = np.stack(
        [compute_step6_v5_per_frame_v6(base_seq[t]) for t in range(T)],
        axis=0,
    )   # (T, 2)

    # ── Step6 v5 spread velocity (2 dims) — derived from scaled step6_v4 ─
    spread_vel = compute_spread_velocity(step6_v4)   # (T, 2)

    return np.concatenate(
        [kinematic, orient, step6_v4, curl, spread_vel], axis=1
    )   # (T, 403)


# ─────────────────────────────────────────────────────────────────────────────
# NORMALISATION  (same as v5)
# ─────────────────────────────────────────────────────────────────────────────

def compute_train_stats(train_windows):
    if not train_windows:
        raise ValueError("No training windows provided.")
    feature_dim = train_windows[0].shape[1]
    flag_idx    = set(_flag_indices())
    coord_mask  = np.ones(feature_dim, dtype=bool)
    for fi in flag_idx:
        coord_mask[fi] = False

    all_frames = np.concatenate(
        [w.reshape(-1, feature_dim) for w in train_windows], axis=0
    ).astype(np.float32)

    mean = np.zeros(feature_dim, dtype=np.float32)
    std  = np.ones(feature_dim,  dtype=np.float32)
    mean[coord_mask] = all_frames[:, coord_mask].mean(axis=0)
    std[coord_mask]  = all_frames[:, coord_mask].std(axis=0) + 1e-6
    for fi in flag_idx:
        mean[fi] = 0.0
        std[fi]  = 1.0
    return {"mean": mean, "std": std}


def normalize_sequence(seq, mean, std):
    return ((seq - mean) / std).astype(np.float32)


# ─────────────────────────────────────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────────────────────────────────────

def main():
    set_seed(SEED)
    SEQ_DIR_V6.mkdir(parents=True, exist_ok=True)
    META_DIR_V6.mkdir(parents=True, exist_ok=True)
    CKPT_DIR_V6.mkdir(parents=True, exist_ok=True)

    print("=" * 62)
    print("BUILD SEQUENCES v6")
    print("=" * 62)
    print(f"  Changes from v5:")
    print(f"    1. Palm-scale normalization for all distance features")
    print(f"    2. Step 6 added to mirror labels (was {{0,2}}, now {{0,2,5}})")
    print(f"  FEATURE_DIM : {FEATURE_DIM}  (unchanged)")
    print(f"  MIRROR_LABELS: {MIRROR_LABELS_V6}")
    print(f"  Output dir  : {OUT_DIR_V6}")

    videos    = list_videos()
    split_map = split_videos(
        videos, LOSO_TEST_SUBJECT, FIXED_VAL_SUBJECT, FIXED_TEST_SUBJECT,
    )
    print(f"\n  Found {len(videos)} videos")

    # ── Pass 1: extract and engineer features ─────────────────────────────
    print(f"\nPass 1/2 — extracting features (FPS={TARGET_FPS})...")

    video_feats        = {}
    video_feats_mirror = {}
    train_windows      = []
    mirror_eligible    = 0

    for i, v in enumerate(tqdm(videos, desc="Extracting")):
        base = extract_base_features(Path(v["video_path"]),
                                     target_fps=TARGET_FPS)
        if base is None:
            video_feats[i] = video_feats_mirror[i] = None
            continue

        feat = build_full_feature_sequence_v6(base)   # (T, 403)
        video_feats[i] = feat

        is_train    = (split_map[i] == "train")
        is_minority = (v["label"] in MIRROR_LABELS_V6)   # now includes Step 6

        if is_train and is_minority:
            base_mirror           = mirror_base_sequence(base)
            feat_mirror           = build_full_feature_sequence_v6(base_mirror)
            video_feats_mirror[i] = feat_mirror
            mirror_eligible      += 1
        else:
            video_feats_mirror[i] = None

        if is_train:
            n = feat.shape[0]
            for start in range(0, n - SEQ_LEN + 1, STRIDE):
                train_windows.append(feat[start: start + SEQ_LEN])
            if video_feats_mirror[i] is not None:
                for start in range(0, n - SEQ_LEN + 1, STRIDE):
                    train_windows.append(
                        video_feats_mirror[i][start: start + SEQ_LEN]
                    )

    print(f"  Collected {len(train_windows)} train windows")
    print(f"  Videos with mirroring : {mirror_eligible}")
    print(f"  Feature dim           : {train_windows[0].shape[1]}")

    # ── Normalisation stats ───────────────────────────────────────────────
    print("  Computing per-feature normalisation statistics...")
    norm_stats = compute_train_stats(train_windows)
    np.savez(META_DIR_V6 / "norm_stats.npz",
             mean=norm_stats["mean"], std=norm_stats["std"])
    print(f"  Saved norm_stats.npz  (dim={norm_stats['mean'].shape[0]})")

    # ── Pass 2: normalize and save ────────────────────────────────────────
    print("\nPass 2/2 — normalising and saving sequences...")

    index     = []
    seq_id    = skipped = n_original = n_mirrored = 0

    for i, v in enumerate(tqdm(videos, desc="Saving")):
        feat = video_feats[i]
        if feat is None:
            skipped += 1
            continue

        n, split = feat.shape[0], split_map[i]

        for start in range(0, n - SEQ_LEN + 1, STRIDE):
            window   = normalize_sequence(
                feat[start: start + SEQ_LEN],
                norm_stats["mean"], norm_stats["std"],
            )
            filename = f"seq_{seq_id:07d}.npy"
            np.save(SEQ_DIR_V6 / filename, window)
            index.append({
                "seq_id":      seq_id,
                "file":        filename,
                "label":       int(v["label"]),
                "video_name":  v["video_name"],
                "subject":     v["subject"],
                "start_frame": int(start),
                "seq_len":     int(SEQ_LEN),
                "feature_dim": int(window.shape[1]),
                "split":       split,
                "mirrored":    False,
            })
            seq_id += 1; n_original += 1

        feat_m = video_feats_mirror[i]
        if feat_m is not None:
            for start in range(0, n - SEQ_LEN + 1, STRIDE):
                window   = normalize_sequence(
                    feat_m[start: start + SEQ_LEN],
                    norm_stats["mean"], norm_stats["std"],
                )
                filename = f"seq_{seq_id:07d}.npy"
                np.save(SEQ_DIR_V6 / filename, window)
                index.append({
                    "seq_id":      seq_id,
                    "file":        filename,
                    "label":       int(v["label"]),
                    "video_name":  v["video_name"] + "_mirror",
                    "subject":     v["subject"],
                    "start_frame": int(start),
                    "seq_len":     int(SEQ_LEN),
                    "feature_dim": int(window.shape[1]),
                    "split":       split,
                    "mirrored":    True,
                })
                seq_id += 1; n_mirrored += 1

    # ── Save metadata ─────────────────────────────────────────────────────
    with open(META_DIR_V6 / "dataset_index.json", "w") as f:
        json.dump(index, f, indent=2)

    split_counts  = {}
    class_counts  = {}
    mirror_counts = {}
    for item in index:
        split_counts[item["split"]] = split_counts.get(item["split"], 0) + 1
        if item["split"] == "train":
            lb = item["label"]
            if item["mirrored"]:
                mirror_counts[lb] = mirror_counts.get(lb, 0) + 1
            else:
                class_counts[lb]  = class_counts.get(lb, 0) + 1

    with open(META_DIR_V6 / "config.json", "w") as f:
        json.dump({
            "version":             "v6_palm_scale_mirror_s6",
            "num_sequences":       len(index),
            "num_original":        n_original,
            "num_mirrored":        n_mirrored,
            "num_videos_skipped":  skipped,
            "feature_dim":         FEATURE_DIM,
            "mirror_labels":       sorted(MIRROR_LABELS_V6),
            "palm_scale_applied":  True,
            "split_counts":        split_counts,
            "feature_dim_layout": {
                "wrist_relative_palm_scaled": "0:128",
                "velocity":                  "128:256",
                "acceleration":              "256:384",
                "orientation_v6":            "384:393",
                "step6_v4_palm_scaled":      "393:399",
                "step6_v5_curl":             "399:401",
                "step6_v5_spread_vel":       "401:403",
            },
        }, f, indent=2)

    # ── Summary ───────────────────────────────────────────────────────────
    print(f"\n{'='*62}")
    print("DATASET READY  (v6)")
    print(f"{'='*62}")
    print(f"  Total sequences : {len(index)}")
    print(f"    Original      : {n_original}")
    print(f"    Mirrored      : {n_mirrored}")
    print(f"  Videos skipped  : {skipped}")
    print(f"  Feature dim     : {FEATURE_DIM}  (unchanged)")
    print(f"  Palm scale      : APPLIED to all distance features")
    print(f"  Split counts    : {split_counts}")

    step_names = ["Step 1","Step 2","Step 3","Step 4","Step 5","Step 6","Step 7"]
    print(f"\n  Train class breakdown:")
    for lb in range(7):
        orig   = class_counts.get(lb, 0)
        mirror = mirror_counts.get(lb, 0)
        tag    = "  ← mirrored" if mirror > 0 else ""
        print(f"    class {lb} {step_names[lb]:8s}: "
              f"{orig:4d} + {mirror:4d} mirror = {orig+mirror:4d} total{tag}")

    print(f"\n  Output → {OUT_DIR_V6}")
    print(f"\n  Next: update config.py paths to v6, then run")
    print(f"  train_teacher_v6.py and train_student_kd_v6.py")


if __name__ == "__main__":
    main()

In [ ]:
%run build_sequences_v6.py

In [ ]:
%%writefile model_tcn_attention.py
"""
model_tcn_attention.py — TCN + BiGRU Attention for HandWash HAR
================================================================
v4 changes (aligned with pipeline v4, FEATURE_DIM = 399):
  - input_dim default = FEATURE_DIM imported live from config
    (currently 399: 384 kinematic + 9 orientation + 6 step6)
  - Stale references to 393 removed from all docstrings
  - All other architecture unchanged from v3:
      BiGRUAttention replaces order-blind TemporalAttention
      FeatureAttention with explicit hidden_dim=96
      use_layer_norm option for deployment safety
      sys.path fix for Kaggle
"""

import sys
_WORKING_DIR = "/kaggle/working"
if _WORKING_DIR not in sys.path:
    sys.path.insert(0, _WORKING_DIR)

import torch
import torch.nn as nn
import torch.nn.functional as F

from config import FEATURE_DIM   # 399  (384 kinematic + 9 orient + 6 step6)


# ─────────────────────────────────────────────────────────────────────────────
# BUILDING BLOCKS
# ─────────────────────────────────────────────────────────────────────────────

class CausalConv1d(nn.Module):
    """
    Left-padded causal 1D convolution.
    Output at time t depends only on inputs at times ≤ t.
    Ensures no future information leaks during the TCN pass.
    """
    def __init__(self, in_ch: int, out_ch: int,
                 kernel_size: int, dilation: int = 1):
        super().__init__()
        self.left_pad = (kernel_size - 1) * dilation
        self.conv     = nn.Conv1d(in_ch, out_ch, kernel_size,
                                  dilation=dilation)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.conv(F.pad(x, (self.left_pad, 0)))


class ResidualTCNBlock(nn.Module):
    """
    Two-layer causal residual block with normalisation and dropout.

    use_layer_norm=True  → GroupNorm(1, C) ≡ LayerNorm on channels.
                           Works correctly at any batch size, including 1.
                           Use this for deployment / real-time inference.
    use_layer_norm=False → BatchNorm1d. Faster during training with
                           large batches. Do NOT use at batch size 1.
    """
    def __init__(self, in_ch: int, out_ch: int,
                 kernel_size: int, dilation: int,
                 dropout: float = 0.2,
                 use_layer_norm: bool = False):
        super().__init__()
        self.conv1 = CausalConv1d(in_ch,  out_ch, kernel_size, dilation)
        self.conv2 = CausalConv1d(out_ch, out_ch, kernel_size, dilation)

        norm_cls   = (lambda c: nn.GroupNorm(1, c)) if use_layer_norm \
                     else nn.BatchNorm1d
        self.norm1 = norm_cls(out_ch)
        self.norm2 = norm_cls(out_ch)

        self.drop = nn.Dropout(dropout)
        self.down = (nn.Conv1d(in_ch, out_ch, 1)
                     if in_ch != out_ch else nn.Identity())

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        res = self.down(x)
        out = self.drop(F.relu(self.norm1(self.conv1(x))))
        out = self.drop(F.relu(self.norm2(self.conv2(out))))
        return F.relu(out + res)


class TCNBranch(nn.Module):
    """
    Stack of ResidualTCNBlocks with exponentially increasing dilations.
    Each dilation doubles the temporal receptive field without adding
    parameters.
    """
    def __init__(self, in_ch: int, channels: int,
                 kernel_size: int, dilations: tuple,
                 dropout: float, use_layer_norm: bool = False):
        super().__init__()
        layers, ch = [], in_ch
        for d in dilations:
            layers.append(
                ResidualTCNBlock(ch, channels, kernel_size, d,
                                 dropout, use_layer_norm)
            )
            ch = channels
        self.net = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class FeatureAttention(nn.Module):
    """
    Channel squeeze-and-excitation attention.

    Globally pools the temporal dimension then learns per-channel
    gating weights in [0, 1] via a small bottleneck MLP.
    Channels consistently important across time are amplified;
    noisy or irrelevant channels are suppressed.

    hidden_dim is set explicitly rather than computed as d_model // k
    to avoid silent integer-floor errors on non-divisible input dims.
    (e.g. 399 // 4 = 99, not a problem per se, but explicit is safer)
    """
    def __init__(self, d_model: int, hidden_dim: int = 96):
        super().__init__()
        self.gate = nn.Sequential(
            nn.AdaptiveAvgPool1d(1),        # (B, C, T) → (B, C, 1)
            nn.Flatten(),                   # (B, C)
            nn.Linear(d_model, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, d_model),
            nn.Sigmoid(),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, C, T)
        return x * self.gate(x).unsqueeze(-1)   # (B, C, T)


class BiGRUAttention(nn.Module):
    """
    Bidirectional GRU followed by additive (Bahdanau) attention pooling.

    Why BiGRU is stronger than simple weighted-sum attention
    ─────────────────────────────────────────────────────────
    Simple attention pools TCN frames independently — it has no concept
    of temporal order.  Two sequences with the same frames in different
    orders produce identical context vectors.  For handwashing this is
    a real weakness since Steps 1-7 are strictly sequential.

    BiGRU attention:
      1. BiGRU processes the TCN output sequence.  Each frame's hidden
         state encodes what came before (forward GRU) AND what comes
         after (backward GRU).
      2. Additive attention weights those order-aware hidden states,
         selecting the most important moments IN temporal context.

    Architecture
    ─────────────
      TCN output  (B, T, fusion_dim)
        → BiGRU   (B, T, gru_hidden * 2)   order-aware hidden states
        → dropout
        → additive attention
        → context vector  (B, gru_hidden * 2)   → classifier

    Parameters
    ──────────
    fusion_dim : input dim from TCN fusion layer
    gru_hidden : hidden units per GRU direction; output dim = gru_hidden * 2
    dropout    : applied to GRU output before attention

    Sizing note:
      With fusion_dim=192 and gru_hidden=128, the context vector is
      256-dim — richer than the 192-dim fusion output while remaining
      well-regularised for a ~2300-sequence training set.
    """

    def __init__(self, fusion_dim: int,
                 gru_hidden: int = 128,
                 dropout: float  = 0.3):
        super().__init__()

        self.gru = nn.GRU(
            input_size  = fusion_dim,
            hidden_size = gru_hidden,
            num_layers  = 1,
            batch_first = True,
            bidirectional = True,
        )
        self.drop = nn.Dropout(dropout)

        attn_dim  = gru_hidden * 2
        self.W    = nn.Linear(attn_dim, attn_dim)
        self.v    = nn.Linear(attn_dim, 1, bias=False)

        self.output_dim = attn_dim   # exposed for classifier sizing

    def forward(self,
                h: torch.Tensor
                ) -> tuple[torch.Tensor, torch.Tensor]:
        """
        Parameters
        ──────────
        h : (B, T, fusion_dim)

        Returns
        ───────
        context : (B, gru_hidden * 2)   attended context vector
        attn    : (B, T)                attention weights
        """
        gru_out, _ = self.gru(h)                               # (B, T, H*2)
        gru_out    = self.drop(gru_out)

        e       = self.v(torch.tanh(self.W(gru_out))).squeeze(-1)   # (B, T)
        attn    = torch.softmax(e, dim=1)                            # (B, T)
        context = torch.sum(gru_out * attn.unsqueeze(-1), dim=1)     # (B, H*2)

        return context, attn


# ─────────────────────────────────────────────────────────────────────────────
# MAIN MODEL
# ─────────────────────────────────────────────────────────────────────────────

class TCNAttentionHAR(nn.Module):
    """
    Multi-branch TCN + BiGRU Attention for WHO handwash step recognition.

    Full forward path
    ─────────────────
      input (B, T, F)                        F = FEATURE_DIM = 399
        → FeatureAttention   optional per-channel gating
        → 3 × TCNBranch      parallel branches, kernels (3, 5, 7)
                             each with dilations (1, 2, 4, 8)
        → cat + 1×1 conv     → (B, T, fusion_dim)
        → BiGRUAttention     order-aware context vector
        → dropout + Linear   → (B, num_classes)

    Auxiliary supervision path (training only)
    ──────────────────────────────────────────
      post-fuse  → frame_cls  → (B, T, num_classes)
      avg over T → CE loss × 0.1  (denser gradient signal per sequence)

    Feature dim layout (399 dims total)
    ────────────────────────────────────
      [0:128]   wrist-relative coords + presence flags
      [128:256] velocity
      [256:384] acceleration
      [384:393] orientation  (palm normals × 2, dot product, z-spread × 2)
      [393:399] step6 disambig (fingertip spread × 4, fingertip-to-wrist × 2)

    Parameters
    ──────────
    input_dim        : feature dim per frame  (default: FEATURE_DIM from config)
    num_classes      : 7  (WHO handwash steps)
    branch_kernels   : kernel sizes for parallel TCN branches
    branch_channels  : output channels per branch
    dilations        : dilation schedule applied in every branch
    fusion_dim       : channels after 1×1 fusion conv
    gru_hidden       : BiGRU hidden units per direction; classifier = gru_hidden*2
    dropout          : applied throughout (TCN blocks, GRU output, final)
    use_feature_attn : enable channel squeeze-excitation at input
    use_layer_norm   : GroupNorm instead of BatchNorm (safer at batch size = 1)
    """

    def __init__(self,
                 input_dim: int        = FEATURE_DIM,
                 num_classes: int      = 7,
                 branch_kernels: tuple = (3, 5, 7),
                 branch_channels: int  = 128,
                 dilations: tuple      = (1, 2, 4, 8),
                 fusion_dim: int       = 192,
                 gru_hidden: int       = 128,
                 dropout: float        = 0.3,
                 use_feature_attn: bool = True,
                 use_layer_norm: bool   = False):
        super().__init__()

        self.fusion_dim       = fusion_dim
        self.use_feature_attn = use_feature_attn

        # ── Channel attention ─────────────────────────────────────────────
        if use_feature_attn:
            self.feat_attn = FeatureAttention(input_dim, hidden_dim=96)

        # ── Parallel TCN branches ─────────────────────────────────────────
        self.branches = nn.ModuleList([
            TCNBranch(input_dim, branch_channels, k, dilations,
                      dropout, use_layer_norm)
            for k in branch_kernels
        ])

        # ── 1×1 fusion conv ───────────────────────────────────────────────
        total_ch = branch_channels * len(branch_kernels)
        self.fuse = nn.Conv1d(total_ch, fusion_dim, kernel_size=1)

        # ── BiGRU attention pooling ───────────────────────────────────────
        self.bigru_attn = BiGRUAttention(
            fusion_dim = fusion_dim,
            gru_hidden = gru_hidden,
            dropout    = dropout,
        )

        classifier_dim = self.bigru_attn.output_dim   # gru_hidden * 2

        # ── Classifiers ───────────────────────────────────────────────────
        self.drop      = nn.Dropout(dropout)
        self.cls       = nn.Linear(classifier_dim, num_classes)

        # Auxiliary frame-level head (used during training only)
        # loss: CE(frame_logits.mean(dim=1), y) * 0.1
        self.frame_cls = nn.Linear(fusion_dim, num_classes)

    def forward(self,
                x: torch.Tensor,
                return_frame_logits: bool = False
                ) -> tuple:
        """
        Parameters
        ──────────
        x                   : (B, T, F)  input sequence
        return_frame_logits : also return per-frame predictions

        Returns
        ───────
        logits       : (B, num_classes)
        attn         : (B, T)               temporal attention weights
        frame_logits : (B, T, num_classes)  only when return_frame_logits=True
        """
        x = x.transpose(1, 2)                              # (B, F, T)

        if self.use_feature_attn:
            x = self.feat_attn(x)

        h   = torch.cat([b(x) for b in self.branches], dim=1)  # (B, total_ch, T)
        h   = self.fuse(h)                                       # (B, fusion_dim, T)
        h_t = h.transpose(1, 2)                                  # (B, T, fusion_dim)

        frame_logits     = self.frame_cls(h_t)              # (B, T, num_classes)
        ctx, attn        = self.bigru_attn(h_t)             # (B, H*2), (B, T)
        logits           = self.cls(self.drop(ctx))         # (B, num_classes)

        if return_frame_logits:
            return logits, frame_logits, attn
        return logits, attn

    def get_intermediate_features(self,
                                   x: torch.Tensor) -> torch.Tensor:
        """
        Post-fusion temporal features for knowledge distillation.

        Returns (B, T, fusion_dim).

        Note: teacher fusion_dim=192, student fusion_dim=144.  A linear
        projection layer must be applied in the KD training script before
        computing the alignment loss.  KD_GAMMA should remain 0.0 until
        that projection is added.
        """
        x = x.transpose(1, 2)
        if self.use_feature_attn:
            x = self.feat_attn(x)
        h = torch.cat([b(x) for b in self.branches], dim=1)
        return self.fuse(h).transpose(1, 2)   # (B, T, fusion_dim)

    def count_parameters(self) -> int:
        """Total trainable parameters."""
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

In [ ]:
%%writefile train_teacher_v6.py
"""
train_teacher_v6.py — Teacher training for v6 pipeline
=======================================================
Completely self-contained — no imports from train_teacher_v5.py.
All required classes and functions are defined here directly so this
file works in a fresh notebook with only config.py and
model_tcn_attention.py present.

Changes from v5:
  1. Paths → handwash_tcn_dataset_v6
  2. Step 6-aware checkpointing: ckpt_score = macro_f1 + 0.4 × s6_recall
  3. CLASS_WEIGHTS[5] = 1.25  (was 1.000)
  4. FOCAL_GAMMA = 1.5        (was 2.0)
  5. STEP6_BLUR_SIGMA = 1.2   (was 0.8)
  6. train_loss + val_loss saved to history
"""

import sys
_WORKING_DIR = "/kaggle/working"
if _WORKING_DIR not in sys.path:
    sys.path.insert(0, _WORKING_DIR)

import json
import random
from pathlib import Path

import numpy as np
from sklearn.metrics import (
    f1_score, accuracy_score, confusion_matrix,
    precision_score, recall_score,
)
from scipy.ndimage import gaussian_filter1d

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

from config import (
    BATCH_SIZE, NUM_WORKERS, WEIGHT_DECAY,
    EARLY_STOP_PATIENCE, GRAD_CLIP,
    TEACHER_BRANCH_KERNELS, TEACHER_BRANCH_CHANNELS,
    TEACHER_DILATIONS, TEACHER_FUSION_DIM, TEACHER_GRU_HIDDEN,
    DROPOUT_TEACHER, FEATURE_DIM, PROTECT_START,
    MIXUP_ALPHA, AUG_ORIENT_PROTECT,
)
from model_tcn_attention import TCNAttentionHAR

# ─────────────────────────────────────────────────────────────────────────────
# V6 PATHS
# ─────────────────────────────────────────────────────────────────────────────

BASE_DIR_V6 = Path("/kaggle/working/handwash_tcn_dataset_v6")
META_DIR_V6 = BASE_DIR_V6 / "metadata"
SEQ_DIR_V6  = BASE_DIR_V6 / "sequences"
CKPT_DIR_V6 = BASE_DIR_V6 / "checkpoints"

# ─────────────────────────────────────────────────────────────────────────────
# V6 HYPERPARAMETERS
# ─────────────────────────────────────────────────────────────────────────────

EPOCHS_TEACHER     = 150
LR_TEACHER         = 1e-3
LABEL_SMOOTHING    = 0.1
NUM_CLASSES        = 7
FOCAL_GAMMA        = 1.5

CLASS_WEIGHTS_V6 = [
    1.414, 1.000, 1.414, 1.000, 1.000,
    1.250,   # Step 6 — was 1.000
    1.000,
]

STEP6_BLUR_SIGMA   = 1.2
STEP6_BLUR_PROB    = 0.7
STEP6_LABEL        = 5
STEP6_RECALL_FLOOR = 0.88
CKPT_S6_WEIGHT     = 0.4

STEP_NAMES = [
    "Rub Palms", "Rub Backs", "Interlace", "Back Fingers",
    "Thumbs", "Fingertips", "Wrists",
]

# ─────────────────────────────────────────────────────────────────────────────
# REPRODUCIBILITY
# ─────────────────────────────────────────────────────────────────────────────

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

# ─────────────────────────────────────────────────────────────────────────────
# FOCAL LOSS
# ─────────────────────────────────────────────────────────────────────────────

class FocalLoss(nn.Module):
    def __init__(self, alpha, gamma=2.0, smoothing=0.1, num_classes=7):
        super().__init__()
        self.register_buffer("alpha", alpha)
        self.gamma       = gamma
        self.smoothing   = smoothing
        self.num_classes = num_classes

    def forward(self, logits, targets):
        B, C     = logits.shape
        log_prob = F.log_softmax(logits, dim=1)
        prob     = log_prob.exp()
        with torch.no_grad():
            sv = self.smoothing / C
            hv = 1.0 - self.smoothing + sv
            ys = torch.full_like(prob, sv)
            ys.scatter_(1, targets.unsqueeze(1), hv)
        p_t = prob.gather(1, targets.unsqueeze(1)).squeeze(1)
        fw  = (1.0 - p_t).pow(self.gamma)
        at  = self.alpha[targets]
        ce  = -(ys * log_prob).sum(dim=1)
        return (at * fw * ce).mean()


def build_criterion(device):
    alpha = torch.tensor(CLASS_WEIGHTS_V6, dtype=torch.float32, device=device)
    return FocalLoss(alpha, gamma=FOCAL_GAMMA,
                     smoothing=LABEL_SMOOTHING, num_classes=NUM_CLASSES)

# ─────────────────────────────────────────────────────────────────────────────
# DATASET
# ─────────────────────────────────────────────────────────────────────────────

class AugmentedSeqDataset(Dataset):
    FLAG_INDICES     = [126, 127, 254, 255, 382, 383]
    STEP6_FEAT_START = 393
    STEP6_FEAT_END   = 403

    def __init__(self, index, split, seq_dir, augment=True):
        self.items   = [x for x in index if x["split"] == split]
        self.seq_dir = Path(seq_dir)
        self.augment = augment and (split == "train")
        if not self.items:
            raise ValueError(f"No samples for split='{split}'")

    def __len__(self): return len(self.items)

    def __getitem__(self, i):
        it = self.items[i]
        x  = np.load(self.seq_dir / it["file"]).astype(np.float32)
        y  = int(it["label"])
        if self.augment:
            x = self._augment(x, y)
        return torch.from_numpy(x), torch.tensor(y, dtype=torch.long)

    def _augment(self, x, label):
        T, F = x.shape
        if np.random.random() < 0.3:
            shift = np.random.randint(-8, 9)
            x = np.roll(x, shift, axis=0)
            if shift > 0:   x[:shift]  = x[shift:shift+1]
            elif shift < 0: x[shift:]  = x[shift-1:shift]
        if np.random.random() < 0.5:
            n = np.random.normal(0, 0.01, x.shape).astype(np.float32)
            n[:, self.FLAG_INDICES] = 0.0
            x = x + n
        if np.random.random() < 0.2:
            m = np.random.binomial(1, 0.95, F).astype(np.float32)
            for fi in self.FLAG_INDICES: m[fi] = 1.0
            if AUG_ORIENT_PROTECT: m[PROTECT_START:] = 1.0
            x = x * m
        if np.random.random() < 0.2:
            ml = np.random.randint(5, max(6, T // 4))
            ms = np.random.randint(0, T - ml)
            x[ms:ms+ml, :] = 0.0
        if np.random.random() < 0.2:
            sc = np.random.uniform(0.9, 1.1)
            x[:, 0:126]   *= sc
            x[:, 128:254] *= sc
            x[:, 256:382] *= sc
        if label == STEP6_LABEL and np.random.random() < STEP6_BLUR_PROB:
            x[:, self.STEP6_FEAT_START:self.STEP6_FEAT_END] = (
                gaussian_filter1d(
                    x[:, self.STEP6_FEAT_START:self.STEP6_FEAT_END],
                    sigma=STEP6_BLUR_SIGMA, axis=0, mode="nearest",
                ).astype(np.float32)
            )
        return x

# ─────────────────────────────────────────────────────────────────────────────
# MIXUP
# ─────────────────────────────────────────────────────────────────────────────

def mixup_batch(x, y, alpha=0.4):
    lam   = float(np.random.beta(alpha, alpha)) if alpha > 0 else 1.0
    index = torch.randperm(x.size(0), device=x.device)
    return lam * x + (1-lam) * x[index], y, y[index], lam

def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1-lam) * criterion(pred, y_b)

# ─────────────────────────────────────────────────────────────────────────────
# TRAIN / EVAL
# ─────────────────────────────────────────────────────────────────────────────

def run_epoch(model, loader, criterion, device,
              optimizer=None, use_mixup=False, aux_weight=0.1):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()
    losses, ys, ps = [], [], []
    ctx = torch.enable_grad() if is_train else torch.no_grad()
    with ctx:
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            if is_train and use_mixup and np.random.random() < 0.5:
                x_mix, ya, yb, lam = mixup_batch(x, y, MIXUP_ALPHA)
                logits, fl, _ = model(x_mix, return_frame_logits=True)
                loss = (mixup_criterion(criterion, logits, ya, yb, lam)
                        + aux_weight * mixup_criterion(criterion, fl.mean(1), ya, yb, lam))
            else:
                logits, fl, _ = model(x, return_frame_logits=True)
                loss = criterion(logits, y) + aux_weight * criterion(fl.mean(1), y)
            if is_train:
                optimizer.zero_grad(); loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                optimizer.step()
            losses.append(loss.item())
            ps.extend(torch.argmax(logits, 1).cpu().numpy().tolist())
            ys.extend(y.cpu().numpy().tolist())
    return {
        "loss": float(np.mean(losses)),
        "acc":  float(accuracy_score(ys, ps)),
        "f1":   float(f1_score(ys, ps, average="macro", zero_division=0)),
    }


def run_epoch_detailed(model, loader, criterion, device):
    model.eval()
    losses, ys, ps = [], [], []
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits, fl, _ = model(x, return_frame_logits=True)
            loss = criterion(logits, y) + 0.1 * criterion(fl.mean(1), y)
            losses.append(loss.item())
            ps.extend(torch.argmax(logits, 1).cpu().numpy().tolist())
            ys.extend(y.cpu().numpy().tolist())
    return {
        "loss":                float(np.mean(losses)),
        "acc":                 float(accuracy_score(ys, ps)),
        "f1":                  float(f1_score(ys, ps, average="macro", zero_division=0)),
        "macro_precision":     float(precision_score(ys, ps, average="macro", zero_division=0)),
        "macro_recall":        float(recall_score(ys, ps, average="macro", zero_division=0)),
        "per_class_f1":        f1_score(ys, ps, average=None, labels=range(NUM_CLASSES), zero_division=0).tolist(),
        "per_class_precision": precision_score(ys, ps, average=None, labels=range(NUM_CLASSES), zero_division=0).tolist(),
        "per_class_recall":    recall_score(ys, ps, average=None, labels=range(NUM_CLASSES), zero_division=0).tolist(),
        "confusion_matrix":    confusion_matrix(ys, ps, labels=range(NUM_CLASSES)).tolist(),
    }

# ─────────────────────────────────────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────────────────────────────────────

def main():
    set_seed(42)
    CKPT_DIR_V6.mkdir(parents=True, exist_ok=True)

    print("="*62)
    print("TEACHER TRAINING  v6  (self-contained)")
    print("="*62)
    print(f"  FEATURE_DIM    : {FEATURE_DIM}")
    print(f"  FOCAL_GAMMA    : {FOCAL_GAMMA}")
    print(f"  S6 weight      : {CLASS_WEIGHTS_V6[5]}")
    print(f"  S6 blur sigma  : {STEP6_BLUR_SIGMA}")
    print(f"  CKPT score     : macro_f1 + {CKPT_S6_WEIGHT} × s6_recall")
    print(f"  Dataset        : {BASE_DIR_V6}")

    with open(META_DIR_V6 / "dataset_index.json") as f:
        index = json.load(f)

    train_ds = AugmentedSeqDataset(index, "train", SEQ_DIR_V6, augment=True)
    val_ds   = AugmentedSeqDataset(index, "val",   SEQ_DIR_V6, augment=False)
    test_ds  = AugmentedSeqDataset(index, "test",  SEQ_DIR_V6, augment=False)

    train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
    val_dl   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)
    test_dl  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

    print(f"\n  Train:{len(train_ds)}  Val:{len(val_ds)}  Test:{len(test_ds)}")
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"  Device: {device}")

    model = TCNAttentionHAR(
        input_dim=FEATURE_DIM, num_classes=NUM_CLASSES,
        branch_kernels=TEACHER_BRANCH_KERNELS,
        branch_channels=TEACHER_BRANCH_CHANNELS,
        dilations=TEACHER_DILATIONS,
        fusion_dim=TEACHER_FUSION_DIM,
        gru_hidden=TEACHER_GRU_HIDDEN,
        dropout=DROPOUT_TEACHER,
        use_feature_attn=True, use_layer_norm=False,
    ).to(device)
    print(f"\n  Parameters : {model.count_parameters():,}")

    criterion = build_criterion(device)
    optimizer = AdamW(model.parameters(), lr=LR_TEACHER, weight_decay=WEIGHT_DECAY)
    scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS_TEACHER, eta_min=1e-6)

    print("\n"+"="*62+"\nTRAINING\n"+"="*62+"\n")

    best_ckpt_score  = -1.0
    best_epoch       = 0
    patience_counter = 0
    history          = []
    s6_floor_active  = False

    for epoch in range(1, EPOCHS_TEACHER + 1):
        tr = run_epoch(model, train_dl, criterion, device,
                       optimizer=optimizer, use_mixup=True, aux_weight=0.1)
        va = run_epoch_detailed(model, val_dl, criterion, device)
        scheduler.step()

        s6_recall  = va["per_class_recall"][STEP6_LABEL]
        ckpt_score = va["f1"] + CKPT_S6_WEIGHT * s6_recall

        print(f"Ep {epoch:03d} | "
              f"Train L:{tr['loss']:.4f} F1:{tr['f1']:.4f} | "
              f"Val L:{va['loss']:.4f} F1:{va['f1']:.4f} | "
              f"Score:{ckpt_score:.4f}")
        print(f"         S6 F1:{va['per_class_f1'][5]:.4f}  "
              f"S6 P:{va['per_class_precision'][5]:.4f}  "
              f"S6 R:{s6_recall:.4f}")

        history.append({
            "epoch":                   epoch,
            "train_loss":              tr["loss"],
            "train_f1":                tr["f1"],
            "val_loss":                va["loss"],
            "val_f1":                  va["f1"],
            "val_ckpt_score":          ckpt_score,
            "val_s6_recall":           s6_recall,
            "val_macro_precision":     va["macro_precision"],
            "val_macro_recall":        va["macro_recall"],
            "val_per_class_f1":        va["per_class_f1"],
            "val_per_class_precision": va["per_class_precision"],
            "val_per_class_recall":    va["per_class_recall"],
            "lr":                      scheduler.get_last_lr()[0],
        })

        if ckpt_score > best_ckpt_score:
            best_ckpt_score = ckpt_score
            best_epoch      = epoch
            patience_counter = 0
            torch.save({
                "model":           model.state_dict(),
                "best_val_f1":     va["f1"],
                "best_ckpt_score": ckpt_score,
                "best_s6_recall":  s6_recall,
                "best_epoch":      epoch,
                "val_per_class_f1":         va["per_class_f1"],
                "val_per_class_precision":  va["per_class_precision"],
                "val_per_class_recall":     va["per_class_recall"],
                "val_confusion_matrix":     va["confusion_matrix"],
                "model_config": {
                    "input_dim":       FEATURE_DIM,
                    "num_classes":     NUM_CLASSES,
                    "branch_kernels":  TEACHER_BRANCH_KERNELS,
                    "branch_channels": TEACHER_BRANCH_CHANNELS,
                    "dilations":       TEACHER_DILATIONS,
                    "fusion_dim":      TEACHER_FUSION_DIM,
                    "gru_hidden":      TEACHER_GRU_HIDDEN,
                    "dropout":         DROPOUT_TEACHER,
                    "focal_gamma":     FOCAL_GAMMA,
                },
            }, CKPT_DIR_V6 / "teacher_best.pt")
            print(f"  ✓ Checkpoint  (score={ckpt_score:.4f}, "
                  f"f1={va['f1']:.4f}, s6r={s6_recall:.4f}, ep={epoch})")
        else:
            patience_counter += 1
            floor_active = STEP6_RECALL_FLOOR > 0.0 and s6_recall < STEP6_RECALL_FLOOR
            if patience_counter >= EARLY_STOP_PATIENCE:
                if floor_active:
                    if not s6_floor_active:
                        print(f"  ⚠  Patience exhausted, S6R={s6_recall:.4f} "
                              f"< floor {STEP6_RECALL_FLOOR} — continuing")
                        s6_floor_active = True
                else:
                    print(f"\n  Early stopping at epoch {epoch}")
                    break
            if not floor_active:
                s6_floor_active = False

    # ── Final test ────────────────────────────────────────────────────────
    print("\n"+"="*62+"\nFINAL TEST EVALUATION\n"+"="*62)
    ckpt = torch.load(CKPT_DIR_V6 / "teacher_best.pt", map_location=device)
    model.load_state_dict(ckpt["model"])
    print(f"  Loaded epoch {ckpt['best_epoch']},  "
          f"score={ckpt['best_ckpt_score']:.4f},  f1={ckpt['best_val_f1']:.4f}")

    test = run_epoch_detailed(model, test_dl, criterion, device)
    print(f"\n  Test macro-F1  : {test['f1']:.4f}")
    print(f"  Test S6 recall : {test['per_class_recall'][5]:.4f}")
    print(f"  Test S6 F1     : {test['per_class_f1'][5]:.4f}")

    print(f"\n  {'Step':<22} {'Prec':>8} {'Recall':>8} {'F1':>8}")
    print("  "+"─"*52)
    for i, name in enumerate(STEP_NAMES):
        flag = (" ← target" if i==5 else " ← minority" if i in (0,2) else "")
        print(f"  Step {i+1} ({name:14s})  "
              f"{test['per_class_precision'][i]:8.4f} "
              f"{test['per_class_recall'][i]:8.4f} "
              f"{test['per_class_f1'][i]:8.4f}{flag}")

    cm = test["confusion_matrix"]
    print(f"\n  Confusion matrix:")
    print("       "+"  ".join(f"P{i}" for i in range(1,8)))
    for i, row in enumerate(cm):
        print(f"    T{i+1}   "+"  ".join(f"{v:3d}" for v in row))

    history.append({
        "epoch": "final_test",
        "test_acc":  test["acc"],  "test_f1": test["f1"],
        "test_macro_precision":     test["macro_precision"],
        "test_macro_recall":        test["macro_recall"],
        "test_per_class_f1":        test["per_class_f1"],
        "test_per_class_precision": test["per_class_precision"],
        "test_per_class_recall":    test["per_class_recall"],
        "test_confusion_matrix":    test["confusion_matrix"],
    })

    with open(CKPT_DIR_V6 / "teacher_history.json", "w") as f:
        json.dump(history, f, indent=2)
    np.save(CKPT_DIR_V6 / "teacher_confusion_matrix.npy",
            np.array(test["confusion_matrix"]))

    print(f"\n  Best epoch      : {best_epoch}")
    print(f"  Best ckpt score : {best_ckpt_score:.4f}")
    print(f"  Test macro-F1   : {test['f1']:.4f}")
    print(f"  Test S6 recall  : {test['per_class_recall'][5]:.4f}")


if __name__ == "__main__":
    main()

In [ ]:
%run train_teacher_v6.py

In [ ]:
%%writefile train_student_kd_v6.py
"""
train_student_kd_v6.py — Student KD training for v6 pipeline
=============================================================
Completely self-contained — no imports from train_teacher_v5.py.
Requires only config.py, model_tcn_attention.py, and train_teacher_v6.py
(for FocalLoss, AugmentedSeqDataset, set_seed which are now defined there).
"""

import sys
_WORKING_DIR = "/kaggle/working"
if _WORKING_DIR not in sys.path:
    sys.path.insert(0, _WORKING_DIR)

import json
import random
from pathlib import Path

import numpy as np
from sklearn.metrics import (
    f1_score, accuracy_score, confusion_matrix,
    precision_score, recall_score,
)

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

from config import (
    BATCH_SIZE, NUM_WORKERS, WEIGHT_DECAY,
    EARLY_STOP_PATIENCE, GRAD_CLIP,
    STUDENT_BRANCH_KERNELS, STUDENT_BRANCH_CHANNELS,
    STUDENT_DILATIONS, STUDENT_FUSION_DIM, STUDENT_GRU_HIDDEN,
    TEACHER_BRANCH_KERNELS, TEACHER_BRANCH_CHANNELS,
    TEACHER_DILATIONS, TEACHER_FUSION_DIM, TEACHER_GRU_HIDDEN,
    DROPOUT_TEACHER, FEATURE_DIM, PROTECT_START,
    MIXUP_ALPHA, AUG_ORIENT_PROTECT, NUM_CLASSES,
)
from model_tcn_attention import TCNAttentionHAR

# Import shared components from train_teacher_v6 (self-contained)
from train_teacher_v6 import (
    AugmentedSeqDataset,
    FocalLoss,
    set_seed,
    run_epoch_detailed,
)

# ─────────────────────────────────────────────────────────────────────────────
# V6 PATHS
# ─────────────────────────────────────────────────────────────────────────────

BASE_DIR_V6 = Path("/kaggle/working/handwash_tcn_dataset_v6")
META_DIR_V6 = BASE_DIR_V6 / "metadata"
SEQ_DIR_V6  = BASE_DIR_V6 / "sequences"
CKPT_DIR_V6 = BASE_DIR_V6 / "checkpoints"

# ─────────────────────────────────────────────────────────────────────────────
# V6 HYPERPARAMETERS
# ─────────────────────────────────────────────────────────────────────────────

KD_T            = 3.0
KD_ALPHA        = 0.4
KD_BETA         = 0.20
KD_GAMMA        = 0.35
AUX_LOSS_WEIGHT = 0.1
FOCAL_GAMMA     = 1.5
LABEL_SMOOTHING = 0.1

CLASS_WEIGHTS_V6 = [
    1.414, 1.000, 1.414, 1.000, 1.000,
    1.250,   # Step 6
    1.000,
]

EPOCHS_STUDENT     = 150
LR_STUDENT         = 8e-4
DROPOUT_STUDENT    = 0.20
STEP6_LABEL        = 5
STEP6_RECALL_FLOOR = 0.88
CKPT_S6_WEIGHT     = 0.4

STEP_NAMES = [
    "Rub Palms", "Rub Backs", "Interlace", "Back Fingers",
    "Thumbs", "Fingertips", "Wrists",
]

# ─────────────────────────────────────────────────────────────────────────────
# KD HELPERS
# ─────────────────────────────────────────────────────────────────────────────

class FeatProjection(nn.Module):
    def __init__(self, student_dim, teacher_dim):
        super().__init__()
        self.proj = nn.Linear(student_dim, teacher_dim, bias=False)
    def forward(self, x): return self.proj(x)


def cosine_feat_loss(projected, target):
    B, T, D = projected.shape
    p = projected.reshape(B*T, D)
    t = target.reshape(B*T, D)
    return (1. - F.cosine_similarity(p, t, dim=1)).mean()


def kd_soft_loss(s_logits, t_logits, T):
    return F.kl_div(F.log_softmax(s_logits/T, dim=1),
                    F.softmax(t_logits/T, dim=1),
                    reduction="batchmean") * T**2


def build_hard_criterion(device):
    alpha = torch.tensor(CLASS_WEIGHTS_V6, dtype=torch.float32, device=device)
    return FocalLoss(alpha, gamma=FOCAL_GAMMA,
                     smoothing=LABEL_SMOOTHING, num_classes=NUM_CLASSES)

# ─────────────────────────────────────────────────────────────────────────────
# TRAIN / EVAL
# ─────────────────────────────────────────────────────────────────────────────

def run_train_epoch(student, teacher, projector, loader,
                    hard_criterion, optimizer, device):
    student.train(); projector.train(); teacher.eval()
    losses, ys, ps = [], [], []

    for x, y in loader:
        x, y = x.to(device), y.to(device)
        with torch.no_grad():
            tl, _ = teacher(x)
            tf     = teacher.get_intermediate_features(x)

        if np.random.random() < 0.5:
            lam  = float(np.random.beta(MIXUP_ALPHA, MIXUP_ALPHA))
            idx  = torch.randperm(x.size(0), device=device)
            xm   = lam*x + (1-lam)*x[idx]
            ya, yb = y, y[idx]
            ts_mix = lam*F.softmax(tl/KD_T,1) + (1-lam)*F.softmax(tl[idx]/KD_T,1)
            with torch.no_grad():
                tfm = teacher.get_intermediate_features(xm)
            sl, sfl, _ = student(xm, return_frame_logits=True)
            sf = student.get_intermediate_features(xm)
            Ls = F.kl_div(F.log_softmax(sl/KD_T,1), ts_mix,
                           reduction="batchmean") * KD_T**2
            Lh = lam*hard_criterion(sl,ya) + (1-lam)*hard_criterion(sl,yb)
            La = lam*hard_criterion(sfl.mean(1),ya) + (1-lam)*hard_criterion(sfl.mean(1),yb)
            Lf = cosine_feat_loss(projector(sf), tfm)
        else:
            sl, sfl, _ = student(x, return_frame_logits=True)
            sf = student.get_intermediate_features(x)
            Ls = kd_soft_loss(sl, tl, KD_T)
            Lh = hard_criterion(sl, y)
            La = hard_criterion(sfl.mean(1), y)
            Lf = cosine_feat_loss(projector(sf), tf)

        loss = KD_ALPHA*Ls + KD_BETA*Lh + KD_GAMMA*Lf + AUX_LOSS_WEIGHT*La
        optimizer.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(
            list(student.parameters()) + list(projector.parameters()), GRAD_CLIP)
        optimizer.step()

        losses.append(loss.item())
        ps.extend(torch.argmax(sl, 1).cpu().numpy().tolist())
        ys.extend(y.cpu().numpy().tolist())

    return {
        "loss": float(np.mean(losses)),
        "acc":  float(accuracy_score(ys, ps)),
        "f1":   float(f1_score(ys, ps, average="macro", zero_division=0)),
    }


def run_eval_epoch_kd(student, teacher, projector, loader,
                      hard_criterion, device, detailed=False):
    student.eval(); projector.eval(); teacher.eval()
    losses, ys, ps = [], [], []

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            tl, _ = teacher(x)
            tf     = teacher.get_intermediate_features(x)
            sl, sfl, _ = student(x, return_frame_logits=True)
            sf = student.get_intermediate_features(x)
            Ls = kd_soft_loss(sl, tl, KD_T)
            Lh = hard_criterion(sl, y)
            La = hard_criterion(sfl.mean(1), y)
            Lf = cosine_feat_loss(projector(sf), tf)
            loss = KD_ALPHA*Ls + KD_BETA*Lh + KD_GAMMA*Lf + AUX_LOSS_WEIGHT*La
            losses.append(loss.item())
            ps.extend(torch.argmax(sl, 1).cpu().numpy().tolist())
            ys.extend(y.cpu().numpy().tolist())

    result = {
        "loss": float(np.mean(losses)),
        "acc":  float(accuracy_score(ys, ps)),
        "f1":   float(f1_score(ys, ps, average="macro", zero_division=0)),
    }
    if detailed:
        result["macro_precision"]     = float(precision_score(ys, ps, average="macro", zero_division=0))
        result["macro_recall"]        = float(recall_score(ys, ps, average="macro", zero_division=0))
        result["per_class_f1"]        = f1_score(ys, ps, average=None, labels=range(NUM_CLASSES), zero_division=0).tolist()
        result["per_class_precision"] = precision_score(ys, ps, average=None, labels=range(NUM_CLASSES), zero_division=0).tolist()
        result["per_class_recall"]    = recall_score(ys, ps, average=None, labels=range(NUM_CLASSES), zero_division=0).tolist()
        result["confusion_matrix"]    = confusion_matrix(ys, ps, labels=range(NUM_CLASSES)).tolist()
    return result

# ─────────────────────────────────────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────────────────────────────────────

def main():
    set_seed(42)
    CKPT_DIR_V6.mkdir(parents=True, exist_ok=True)

    print("="*62)
    print("STUDENT KD TRAINING  v6  (self-contained)")
    print("="*62)
    print(f"  FEATURE_DIM     : {FEATURE_DIM}")
    print(f"  FOCAL_GAMMA     : {FOCAL_GAMMA}")
    print(f"  KD T/α/β/γ      : {KD_T}/{KD_ALPHA}/{KD_BETA}/{KD_GAMMA}")
    print(f"  DROPOUT_STUDENT : {DROPOUT_STUDENT}")
    print(f"  S6 weight       : {CLASS_WEIGHTS_V6[5]}")
    print(f"  CKPT score      : macro_f1 + {CKPT_S6_WEIGHT} × s6_recall")
    print(f"  Dataset         : {BASE_DIR_V6}")

    with open(META_DIR_V6 / "dataset_index.json") as f:
        index = json.load(f)

    train_ds = AugmentedSeqDataset(index, "train", SEQ_DIR_V6, augment=True)
    val_ds   = AugmentedSeqDataset(index, "val",   SEQ_DIR_V6, augment=False)
    test_ds  = AugmentedSeqDataset(index, "test",  SEQ_DIR_V6, augment=False)

    train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
    val_dl   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)
    test_dl  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

    print(f"\n  Train:{len(train_ds)}  Val:{len(val_ds)}  Test:{len(test_ds)}")
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"  Device: {device}")

    # ── Teacher ───────────────────────────────────────────────────────────
    teacher = TCNAttentionHAR(
        input_dim=FEATURE_DIM, num_classes=NUM_CLASSES,
        branch_kernels=TEACHER_BRANCH_KERNELS,
        branch_channels=TEACHER_BRANCH_CHANNELS,
        dilations=TEACHER_DILATIONS,
        fusion_dim=TEACHER_FUSION_DIM,
        gru_hidden=TEACHER_GRU_HIDDEN,
        dropout=DROPOUT_TEACHER,
        use_feature_attn=True, use_layer_norm=False,
    ).to(device)

    t_ckpt_path = CKPT_DIR_V6 / "teacher_best.pt"
    if not t_ckpt_path.exists():
        raise FileNotFoundError(
            f"Teacher checkpoint not found: {t_ckpt_path}\n"
            "Run train_teacher_v6.py first.")
    t_ckpt = torch.load(t_ckpt_path, map_location=device)
    teacher.load_state_dict(t_ckpt["model"])
    teacher.eval()
    for p in teacher.parameters(): p.requires_grad_(False)
    print(f"\n  Teacher: ep={t_ckpt['best_epoch']}  "
          f"val_f1={t_ckpt['best_val_f1']:.4f}  "
          f"s6r={t_ckpt['best_s6_recall']:.4f}")

    # ── Student ───────────────────────────────────────────────────────────
    student = TCNAttentionHAR(
        input_dim=FEATURE_DIM, num_classes=NUM_CLASSES,
        branch_kernels=STUDENT_BRANCH_KERNELS,
        branch_channels=STUDENT_BRANCH_CHANNELS,
        dilations=STUDENT_DILATIONS,
        fusion_dim=STUDENT_FUSION_DIM,
        gru_hidden=STUDENT_GRU_HIDDEN,
        dropout=DROPOUT_STUDENT,
        use_feature_attn=True, use_layer_norm=False,
    ).to(device)

    projector  = FeatProjection(STUDENT_FUSION_DIM, TEACHER_FUSION_DIM).to(device)
    s_params   = student.count_parameters()
    t_params   = sum(p.numel() for p in teacher.parameters())
    p_params   = sum(p.numel() for p in projector.parameters())
    print(f"  Student  : {s_params:,} params  ({s_params/t_params*100:.1f}% of teacher)")
    print(f"  Projector: {p_params:,} params")

    hard_criterion = build_hard_criterion(device)
    all_params     = list(student.parameters()) + list(projector.parameters())
    optimizer      = AdamW(all_params, lr=LR_STUDENT, weight_decay=WEIGHT_DECAY)
    scheduler      = CosineAnnealingLR(optimizer, T_max=EPOCHS_STUDENT, eta_min=1e-6)

    print("\n"+"="*62+"\nTRAINING\n"+"="*62+"\n")

    best_ckpt_score  = -1.0
    best_epoch       = 0
    patience_counter = 0
    history          = []
    s6_floor_active  = False

    for epoch in range(1, EPOCHS_STUDENT + 1):
        tr = run_train_epoch(student, teacher, projector,
                             train_dl, hard_criterion, optimizer, device)
        va = run_eval_epoch_kd(student, teacher, projector,
                               val_dl, hard_criterion, device, detailed=True)
        scheduler.step()

        s6_recall  = va["per_class_recall"][STEP6_LABEL]
        ckpt_score = va["f1"] + CKPT_S6_WEIGHT * s6_recall

        print(f"Ep {epoch:03d} | "
              f"Train L:{tr['loss']:.4f} F1:{tr['f1']:.4f} | "
              f"Val L:{va['loss']:.4f} F1:{va['f1']:.4f} | "
              f"Score:{ckpt_score:.4f}")
        print(f"         S6 F1:{va['per_class_f1'][5]:.4f}  "
              f"S6 P:{va['per_class_precision'][5]:.4f}  "
              f"S6 R:{s6_recall:.4f}")

        history.append({
            "epoch":                   epoch,
            "train_loss":              tr["loss"],
            "train_f1":                tr["f1"],
            "val_loss":                va["loss"],
            "val_f1":                  va["f1"],
            "val_ckpt_score":          ckpt_score,
            "val_s6_recall":           s6_recall,
            "val_macro_precision":     va["macro_precision"],
            "val_macro_recall":        va["macro_recall"],
            "val_per_class_f1":        va["per_class_f1"],
            "val_per_class_precision": va["per_class_precision"],
            "val_per_class_recall":    va["per_class_recall"],
            "lr":                      scheduler.get_last_lr()[0],
        })

        if ckpt_score > best_ckpt_score:
            best_ckpt_score = ckpt_score
            best_epoch      = epoch
            patience_counter = 0
            torch.save({
                "model":           student.state_dict(),
                "projector":       projector.state_dict(),
                "best_val_f1":     va["f1"],
                "best_ckpt_score": ckpt_score,
                "best_s6_recall":  s6_recall,
                "best_epoch":      epoch,
                "val_per_class_f1":         va["per_class_f1"],
                "val_per_class_precision":  va["per_class_precision"],
                "val_per_class_recall":     va["per_class_recall"],
                "val_confusion_matrix":     va["confusion_matrix"],
                "model_config": {
                    "input_dim":       FEATURE_DIM,
                    "num_classes":     NUM_CLASSES,
                    "branch_kernels":  STUDENT_BRANCH_KERNELS,
                    "branch_channels": STUDENT_BRANCH_CHANNELS,
                    "dilations":       STUDENT_DILATIONS,
                    "fusion_dim":      STUDENT_FUSION_DIM,
                    "gru_hidden":      STUDENT_GRU_HIDDEN,
                    "dropout":         DROPOUT_STUDENT,
                    "focal_gamma":     FOCAL_GAMMA,
                },
                "kd_config": {
                    "T": KD_T, "alpha": KD_ALPHA, "beta": KD_BETA,
                    "gamma": KD_GAMMA, "ckpt_s6_weight": CKPT_S6_WEIGHT,
                },
            }, CKPT_DIR_V6 / "student_best.pt")
            print(f"  ✓ Checkpoint  (score={ckpt_score:.4f}, "
                  f"f1={va['f1']:.4f}, s6r={s6_recall:.4f}, ep={epoch})")
        else:
            patience_counter += 1
            floor_active = STEP6_RECALL_FLOOR > 0.0 and s6_recall < STEP6_RECALL_FLOOR
            if patience_counter >= EARLY_STOP_PATIENCE:
                if floor_active:
                    if not s6_floor_active:
                        print(f"  ⚠  Patience exhausted, S6R={s6_recall:.4f} "
                              f"< floor {STEP6_RECALL_FLOOR} — continuing")
                        s6_floor_active = True
                else:
                    print(f"\n  Early stopping at epoch {epoch}")
                    break
            if not floor_active:
                s6_floor_active = False

    # ── Final test ────────────────────────────────────────────────────────
    print("\n"+"="*62+"\nFINAL TEST EVALUATION\n"+"="*62)
    ckpt_s = torch.load(CKPT_DIR_V6 / "student_best.pt", map_location=device)
    student.load_state_dict(ckpt_s["model"])
    projector.load_state_dict(ckpt_s["projector"])
    print(f"  Loaded ep={ckpt_s['best_epoch']}  "
          f"score={ckpt_s['best_ckpt_score']:.4f}  "
          f"f1={ckpt_s['best_val_f1']:.4f}  "
          f"s6r={ckpt_s['best_s6_recall']:.4f}")

    test = run_eval_epoch_kd(student, teacher, projector,
                             test_dl, hard_criterion, device, detailed=True)

    print(f"\n  Test macro-F1  : {test['f1']:.4f}")
    print(f"  Test S6 recall : {test['per_class_recall'][5]:.4f}")
    print(f"  Test S6 F1     : {test['per_class_f1'][5]:.4f}")

    print(f"\n  {'Step':<22} {'Prec':>8} {'Recall':>8} {'F1':>8}")
    print("  "+"─"*52)
    for i, name in enumerate(STEP_NAMES):
        flag = (" ← target" if i==5 else " ← minority" if i in (0,2) else "")
        print(f"  Step {i+1} ({name:14s})  "
              f"{test['per_class_precision'][i]:8.4f} "
              f"{test['per_class_recall'][i]:8.4f} "
              f"{test['per_class_f1'][i]:8.4f}{flag}")

    cm = test["confusion_matrix"]
    print(f"\n  Confusion matrix:")
    print("       "+"  ".join(f"P{i}" for i in range(1,8)))
    for i, row in enumerate(cm):
        print(f"    T{i+1}   "+"  ".join(f"{v:3d}" for v in row))

    history.append({
        "epoch": "final_test",
        "test_acc":  test["acc"],  "test_f1": test["f1"],
        "test_macro_precision":     test["macro_precision"],
        "test_macro_recall":        test["macro_recall"],
        "test_per_class_f1":        test["per_class_f1"],
        "test_per_class_precision": test["per_class_precision"],
        "test_per_class_recall":    test["per_class_recall"],
        "test_confusion_matrix":    test["confusion_matrix"],
    })

    with open(CKPT_DIR_V6 / "student_history.json", "w") as f:
        json.dump(history, f, indent=2)
    np.save(CKPT_DIR_V6 / "student_confusion_matrix.npy",
            np.array(test["confusion_matrix"]))

    print(f"\n  Teacher : {t_params:,} params")
    print(f"  Student : {s_params:,} params  ({s_params/t_params*100:.1f}% of teacher)")
    print(f"\n  Best epoch      : {best_epoch}")
    print(f"  Best ckpt score : {best_ckpt_score:.4f}")
    print(f"  Test macro-F1   : {test['f1']:.4f}")
    print(f"  Test S6 recall  : {test['per_class_recall'][5]:.4f}")


if __name__ == "__main__":
    main()

In [ ]:
%run train_student_kd_v6.py

In [ ]:
%%writefile evaluate_student_final.py
"""
evaluate_student_final.py — Full evaluation suite for the locked-in student model
======================================================================================
Generates every standard evaluation artifact for student_best.pt
(best_epoch=30, val_f1=0.9848, s6r=1.0, test_f1=0.9731, test_s6r=0.85):

  1. Training/validation loss curves (train_loss, val_loss vs epoch)
  2. Training/validation F1 curves (train_f1, val_f1 vs epoch)
  3. Checkpoint score curve with the selected best-epoch marked
  4. S6 recall/precision/F1 trajectory across epochs (the target metric)
  5. Confusion matrix heatmap (test set, raw counts + normalised %)
  6. Per-class precision/recall/F1 bar chart (test set)
  7. Test vs Validation comparison bar chart (sanity check for overfitting)
  8. Learning rate schedule plot
  9. Summary text panel with all key numbers in one place

All plots saved to /kaggle/working/handwash_tcn_dataset_v6/evaluation/
and displayed inline in the notebook.

Usage
-----
exec(open("/kaggle/working/evaluate_student_final.py").read())
run_full_evaluation()
"""

import json
import sys
from pathlib import Path

import numpy as np
import torch
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

try:
    from IPython.display import display, Image as IPImage
    from io import BytesIO
    _IN_NOTEBOOK = True
except ImportError:
    _IN_NOTEBOOK = False

sys.path.insert(0, "/kaggle/working")

# ─────────────────────────────────────────────────────────────────────────────
# PATHS
# ─────────────────────────────────────────────────────────────────────────────

BASE_DIR    = Path("/kaggle/working/handwash_tcn_dataset_v6")
CKPT_PATH   = BASE_DIR / "checkpoints" / "student_best.pt"
HISTORY_PATH= BASE_DIR / "checkpoints" / "student_history.json"
OUT_DIR     = BASE_DIR / "evaluation"
OUT_DIR.mkdir(parents=True, exist_ok=True)

STEP_NAMES = [
    "Step 1\nRub Palms",    "Step 2\nRub Backs",
    "Step 3\nInterlace",    "Step 4\nBack Fingers",
    "Step 5\nThumbs",       "Step 6\nFingertips",
    "Step 7\nWrists",
]
STEP_SHORT  = ["S1","S2","S3","S4","S5","S6","S7"]
STEP_COLORS = ["#3B82F6","#10B981","#F59E0B","#EF4444",
               "#8B5CF6","#EC4899","#14B8A6"]
NUM_CLASSES = 7


def _show(fig, dpi=120):
    fig.tight_layout()
    if _IN_NOTEBOOK:
        buf = BytesIO()
        fig.savefig(buf, format="png", bbox_inches="tight", dpi=dpi)
        buf.seek(0)
        plt.close(fig)
        display(IPImage(data=buf.read()))
    else:
        plt.close(fig)


def _save_and_show(fig, name, dpi=130):
    path = OUT_DIR / name
    fig.savefig(path, bbox_inches="tight", dpi=dpi)
    print(f"  Saved → {path}")
    _show(fig)


# ─────────────────────────────────────────────────────────────────────────────
# LOAD DATA
# ─────────────────────────────────────────────────────────────────────────────

def load_checkpoint_and_history():
    if not CKPT_PATH.exists():
        raise FileNotFoundError(f"Checkpoint not found: {CKPT_PATH}")
    if not HISTORY_PATH.exists():
        raise FileNotFoundError(f"History not found: {HISTORY_PATH}")

    ckpt = torch.load(CKPT_PATH, map_location="cpu", weights_only=False)
    with open(HISTORY_PATH) as f:
        history = json.load(f)

    # Separate per-epoch records from the final "final_test" summary record
    epoch_records = [h for h in history if h.get("epoch") != "final_test"]
    test_record   = next((h for h in history if h.get("epoch") == "final_test"), None)

    return ckpt, epoch_records, test_record


# ─────────────────────────────────────────────────────────────────────────────
# 1. LOSS CURVES
# ─────────────────────────────────────────────────────────────────────────────

def plot_loss_curves(epoch_records, best_epoch):
    epochs = [h["epoch"] for h in epoch_records]
    train_loss = [h["train_loss"] for h in epoch_records]
    val_loss   = [h["val_loss"]   for h in epoch_records]

    fig, ax = plt.subplots(figsize=(10, 5.5))
    ax.plot(epochs, train_loss, label="Train Loss", color="#3B82F6", lw=1.8)
    ax.plot(epochs, val_loss,   label="Val Loss",   color="#EF4444", lw=1.8)
    ax.axvline(best_epoch, color="#10B981", ls="--", lw=2,
              label=f"Best checkpoint (epoch {best_epoch})")
    ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
    ax.set_title("Training & Validation Loss Curves", fontweight="bold", fontsize=13)
    ax.legend(fontsize=10)
    ax.grid(alpha=0.3)
    _save_and_show(fig, "01_loss_curves.png")


# ─────────────────────────────────────────────────────────────────────────────
# 2. F1 CURVES
# ─────────────────────────────────────────────────────────────────────────────

def plot_f1_curves(epoch_records, best_epoch, best_val_f1):
    epochs = [h["epoch"] for h in epoch_records]
    train_f1 = [h["train_f1"] for h in epoch_records]
    val_f1   = [h["val_f1"]   for h in epoch_records]

    fig, ax = plt.subplots(figsize=(10, 5.5))
    ax.plot(epochs, train_f1, label="Train F1", color="#3B82F6", lw=1.8)
    ax.plot(epochs, val_f1,   label="Val F1",   color="#EF4444", lw=1.8)
    ax.axvline(best_epoch, color="#10B981", ls="--", lw=2,
              label=f"Best checkpoint (epoch {best_epoch})")
    ax.scatter([best_epoch], [best_val_f1], color="#10B981", s=90, zorder=5)
    ax.annotate(f"{best_val_f1:.4f}", (best_epoch, best_val_f1),
               textcoords="offset points", xytext=(10,-12), fontsize=10,
               fontweight="bold", color="#10B981")
    ax.set_xlabel("Epoch"); ax.set_ylabel("Macro F1")
    ax.set_ylim(0, 1.05)
    ax.set_title("Training & Validation F1 Curves", fontweight="bold", fontsize=13)
    ax.legend(fontsize=10, loc="lower right")
    ax.grid(alpha=0.3)
    _save_and_show(fig, "02_f1_curves.png")


# ─────────────────────────────────────────────────────────────────────────────
# 3. CHECKPOINT SCORE CURVE
# ─────────────────────────────────────────────────────────────────────────────

def plot_ckpt_score_curve(epoch_records, best_epoch, best_score):
    epochs = [h["epoch"] for h in epoch_records]
    scores = [h["val_ckpt_score"] for h in epoch_records]

    fig, ax = plt.subplots(figsize=(10, 5.5))
    ax.plot(epochs, scores, color="#8B5CF6", lw=1.8)
    ax.axvline(best_epoch, color="#10B981", ls="--", lw=2)
    ax.scatter([best_epoch], [best_score], color="#10B981", s=110, zorder=5,
              label=f"Selected checkpoint: epoch {best_epoch}, score={best_score:.4f}")
    ax.set_xlabel("Epoch"); ax.set_ylabel("Checkpoint Score\n(val_f1 + 0.4 × s6_recall)")
    ax.set_title("Checkpoint Selection Score Over Training", fontweight="bold", fontsize=13)
    ax.legend(fontsize=10, loc="lower right")
    ax.grid(alpha=0.3)
    _save_and_show(fig, "03_checkpoint_score_curve.png")


# ─────────────────────────────────────────────────────────────────────────────
# 4. S6 RECALL/PRECISION/F1 TRAJECTORY
# ─────────────────────────────────────────────────────────────────────────────

def plot_s6_trajectory(epoch_records, best_epoch):
    epochs = [h["epoch"] for h in epoch_records]
    s6_recall = [h["val_per_class_recall"][5] for h in epoch_records]
    s6_prec   = [h["val_per_class_precision"][5] for h in epoch_records]
    s6_f1     = [h["val_per_class_f1"][5] for h in epoch_records]

    fig, ax = plt.subplots(figsize=(10, 5.5))
    ax.plot(epochs, s6_recall, label="S6 Recall",    color="#EC4899", lw=2)
    ax.plot(epochs, s6_prec,   label="S6 Precision", color="#F59E0B", lw=1.6, alpha=0.8)
    ax.plot(epochs, s6_f1,     label="S6 F1",        color="#8B5CF6", lw=1.6, alpha=0.8)
    ax.axvline(best_epoch, color="#10B981", ls="--", lw=2,
              label=f"Best checkpoint (epoch {best_epoch})")
    ax.axhline(0.88, color="grey", ls=":", lw=1.3, label="S6 recall floor (0.88)")
    ax.set_xlabel("Epoch"); ax.set_ylabel("Score")
    ax.set_ylim(0, 1.05)
    ax.set_title("Step 6 (Fingertips) Metrics Over Training — The Target Class",
                fontweight="bold", fontsize=13)
    ax.legend(fontsize=9, loc="lower right")
    ax.grid(alpha=0.3)
    _save_and_show(fig, "04_s6_trajectory.png")


# ─────────────────────────────────────────────────────────────────────────────
# 5. CONFUSION MATRIX  (raw + normalised)
# ─────────────────────────────────────────────────────────────────────────────

def plot_confusion_matrix(test_record):
    cm = np.array(test_record["test_confusion_matrix"])
    cm_norm = cm / cm.sum(axis=1, keepdims=True)

    fig, axes = plt.subplots(1, 2, figsize=(15, 6.5))

    # Raw counts
    ax = axes[0]
    im = ax.imshow(cm, cmap="Blues")
    for i in range(NUM_CLASSES):
        for j in range(NUM_CLASSES):
            color = "white" if cm[i,j] > cm.max()/2 else "black"
            ax.text(j, i, str(cm[i,j]), ha="center", va="center",
                   color=color, fontsize=11, fontweight="bold")
    ax.set_xticks(range(NUM_CLASSES)); ax.set_yticks(range(NUM_CLASSES))
    ax.set_xticklabels(STEP_SHORT); ax.set_yticklabels(STEP_SHORT)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    ax.set_title("Confusion Matrix — Raw Counts", fontweight="bold")
    plt.colorbar(im, ax=ax, fraction=0.046)

    # Normalised %
    ax = axes[1]
    im = ax.imshow(cm_norm, cmap="Blues", vmin=0, vmax=1)
    for i in range(NUM_CLASSES):
        for j in range(NUM_CLASSES):
            color = "white" if cm_norm[i,j] > 0.5 else "black"
            ax.text(j, i, f"{cm_norm[i,j]*100:.0f}%", ha="center", va="center",
                   color=color, fontsize=10, fontweight="bold")
    ax.set_xticks(range(NUM_CLASSES)); ax.set_yticks(range(NUM_CLASSES))
    ax.set_xticklabels(STEP_SHORT); ax.set_yticklabels(STEP_SHORT)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    ax.set_title("Confusion Matrix — Row-Normalised %\n(= Recall per class)",
                fontweight="bold")
    plt.colorbar(im, ax=ax, fraction=0.046)

    fig.suptitle("Test Set Confusion Matrix (n=328)", fontsize=14, fontweight="bold", y=1.02)
    _save_and_show(fig, "05_confusion_matrix.png")


# ─────────────────────────────────────────────────────────────────────────────
# 6. PER-CLASS PRECISION/RECALL/F1 BAR CHART
# ─────────────────────────────────────────────────────────────────────────────

def plot_per_class_metrics(test_record):
    prec = test_record["test_per_class_precision"]
    rec  = test_record["test_per_class_recall"]
    f1   = test_record["test_per_class_f1"]

    x = np.arange(NUM_CLASSES)
    width = 0.25

    fig, ax = plt.subplots(figsize=(12, 6))
    b1 = ax.bar(x - width, prec, width, label="Precision", color="#3B82F6")
    b2 = ax.bar(x,         rec,  width, label="Recall",    color="#EF4444")
    b3 = ax.bar(x + width, f1,   width, label="F1",        color="#10B981")

    for bars in [b1, b2, b3]:
        for bar in bars:
            h = bar.get_height()
            ax.text(bar.get_x()+bar.get_width()/2, h+0.015, f"{h:.2f}",
                   ha="center", fontsize=8)

    ax.set_xticks(x)
    ax.set_xticklabels(STEP_SHORT)
    ax.set_ylim(0, 1.15)
    ax.set_ylabel("Score")
    ax.set_title("Test Set — Per-Class Precision, Recall, F1", fontweight="bold", fontsize=13)
    ax.legend(fontsize=10)
    ax.grid(axis="y", alpha=0.3)

    # Highlight S6 as the target class
    ax.axvspan(4.6, 5.4, color="#EC4899", alpha=0.08, zorder=0)
    ax.text(5, 1.10, "target class", ha="center", fontsize=8,
           color="#EC4899", fontweight="bold")

    _save_and_show(fig, "06_per_class_metrics.png")


# ─────────────────────────────────────────────────────────────────────────────
# 7. VAL vs TEST COMPARISON (overfitting sanity check)
# ─────────────────────────────────────────────────────────────────────────────

def plot_val_vs_test(ckpt, test_record):
    val_recall = ckpt["val_per_class_recall"]
    test_recall = test_record["test_per_class_recall"]

    x = np.arange(NUM_CLASSES)
    width = 0.35

    fig, ax = plt.subplots(figsize=(11, 5.5))
    ax.bar(x - width/2, val_recall,  width, label="Validation Recall", color="#8B5CF6")
    ax.bar(x + width/2, test_recall, width, label="Test Recall",       color="#F59E0B")
    ax.set_xticks(x); ax.set_xticklabels(STEP_SHORT)
    ax.set_ylim(0, 1.15)
    ax.set_ylabel("Recall")
    ax.set_title("Validation vs Test Recall — Generalisation Check",
                fontweight="bold", fontsize=13)
    ax.legend(fontsize=10)
    ax.grid(axis="y", alpha=0.3)

    gap = np.array(val_recall) - np.array(test_recall)
    max_gap_idx = int(np.argmax(np.abs(gap)))
    ax.annotate(f"Largest val→test gap:\n{STEP_SHORT[max_gap_idx]} "
               f"({gap[max_gap_idx]:+.2f})",
               xy=(max_gap_idx, max(val_recall[max_gap_idx], test_recall[max_gap_idx])),
               xytext=(max_gap_idx+0.8, 1.05),
               fontsize=9, ha="left",
               arrowprops=dict(arrowstyle="->", color="grey"))

    _save_and_show(fig, "07_val_vs_test.png")


# ─────────────────────────────────────────────────────────────────────────────
# 8. LEARNING RATE SCHEDULE
# ─────────────────────────────────────────────────────────────────────────────

def plot_lr_schedule(epoch_records):
    if "lr" not in epoch_records[0]:
        print("  (No LR data in history — skipping)")
        return
    epochs = [h["epoch"] for h in epoch_records]
    lrs    = [h["lr"] for h in epoch_records]

    fig, ax = plt.subplots(figsize=(10, 4.5))
    ax.plot(epochs, lrs, color="#14B8A6", lw=1.8)
    ax.set_xlabel("Epoch"); ax.set_ylabel("Learning Rate")
    ax.set_title("Cosine Annealing LR Schedule", fontweight="bold", fontsize=13)
    ax.grid(alpha=0.3)
    _save_and_show(fig, "08_lr_schedule.png")


# ─────────────────────────────────────────────────────────────────────────────
# 9. SUMMARY PANEL
# ─────────────────────────────────────────────────────────────────────────────

def plot_summary_panel(ckpt, test_record):
    fig, ax = plt.subplots(figsize=(11, 7.5))
    ax.axis("off")

    lines = [
        ("MODEL SUMMARY", "header"),
        (f"Checkpoint: student_best.pt  (epoch {ckpt['best_epoch']})", "normal"),
        (f"KD config: T={ckpt['kd_config']['T']}  α={ckpt['kd_config']['alpha']}  "
         f"β={ckpt['kd_config']['beta']}  γ={ckpt['kd_config']['gamma']}", "normal"),
        ("", "normal"),
        ("VALIDATION METRICS (best checkpoint)", "header"),
        (f"  Macro F1        : {ckpt['best_val_f1']:.4f}", "normal"),
        (f"  S6 Recall       : {ckpt['best_s6_recall']:.4f}", "normal"),
        (f"  Checkpoint score: {ckpt['best_ckpt_score']:.4f}", "normal"),
        ("", "normal"),
        ("TEST SET METRICS (held-out, n=328)", "header"),
        (f"  Macro F1        : {test_record['test_f1']:.4f}", "normal"),
        (f"  Accuracy        : {test_record['test_acc']:.4f}", "normal"),
        (f"  Macro Precision : {test_record['test_macro_precision']:.4f}", "normal"),
        (f"  Macro Recall    : {test_record['test_macro_recall']:.4f}", "normal"),
        (f"  S6 (target) Recall: {test_record['test_per_class_recall'][5]:.4f}", "highlight"),
        (f"  S6 (target) F1    : {test_record['test_per_class_f1'][5]:.4f}", "highlight"),
        ("", "normal"),
        ("PER-CLASS TEST RECALL", "header"),
    ]
    for i, name in enumerate(STEP_SHORT):
        r = test_record["test_per_class_recall"][i]
        flag = "  ← target" if i == 5 else ""
        lines.append((f"  {name}: {r:.4f}{flag}", "normal"))

    y = 0.97
    for text, style in lines:
        if style == "header":
            ax.text(0.02, y, text, fontsize=13, fontweight="bold",
                   color="#1e293b", transform=ax.transAxes)
            y -= 0.052
        elif style == "highlight":
            ax.text(0.02, y, text, fontsize=11, fontweight="bold",
                   color="#EC4899", transform=ax.transAxes)
            y -= 0.042
        else:
            ax.text(0.02, y, text, fontsize=11,
                   color="#334155", transform=ax.transAxes)
            y -= 0.042

    ax.set_title("Final Model Summary", fontweight="bold", fontsize=15, pad=15)
    _save_and_show(fig, "09_summary_panel.png")


# ─────────────────────────────────────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────────────────────────────────────

def run_full_evaluation():
    print("="*62)
    print("FULL EVALUATION SUITE — student_best.pt")
    print("="*62)

    ckpt, epoch_records, test_record = load_checkpoint_and_history()

    print(f"\n  Checkpoint: epoch {ckpt['best_epoch']}, "
          f"val_f1={ckpt['best_val_f1']:.4f}, "
          f"s6_recall={ckpt['best_s6_recall']:.4f}")
    print(f"  Test:       f1={test_record['test_f1']:.4f}, "
          f"s6_recall={test_record['test_per_class_recall'][5]:.4f}")
    print(f"  Epoch records: {len(epoch_records)}")

    best_epoch = ckpt["best_epoch"]
    best_score = ckpt["best_ckpt_score"]
    best_val_f1= ckpt["best_val_f1"]

    print(f"\n[1/9] Loss curves...")
    plot_loss_curves(epoch_records, best_epoch)

    print(f"\n[2/9] F1 curves...")
    plot_f1_curves(epoch_records, best_epoch, best_val_f1)

    print(f"\n[3/9] Checkpoint score curve...")
    plot_ckpt_score_curve(epoch_records, best_epoch, best_score)

    print(f"\n[4/9] S6 trajectory...")
    plot_s6_trajectory(epoch_records, best_epoch)

    print(f"\n[5/9] Confusion matrix...")
    plot_confusion_matrix(test_record)

    print(f"\n[6/9] Per-class metrics...")
    plot_per_class_metrics(test_record)

    print(f"\n[7/9] Val vs Test comparison...")
    plot_val_vs_test(ckpt, test_record)

    print(f"\n[8/9] LR schedule...")
    plot_lr_schedule(epoch_records)

    print(f"\n[9/9] Summary panel...")
    plot_summary_panel(ckpt, test_record)

    print(f"\n{'='*62}")
    print(f"ALL PLOTS SAVED → {OUT_DIR}")
    print(f"{'='*62}")


if __name__ == "__main__":
    run_full_evaluation()

In [ ]:
%run evaluate_student_final.py

In [ ]:
%%writefile test_continuous_sequence.py
"""
test_continuous_sequence.py — Test v6 model on a full WHO 7-step sequence
============================================================================
For testing a single continuous video that performs all 7 WHO handwash
steps IN ORDER (Step 1 -> Step 2 -> ... -> Step 7), matching the WHO
guideline that the full sequence takes ~40-60 seconds.

Ground truth assumption
------------------------
The 7 steps are assumed to occur strictly in order. You provide EITHER:
  (a) equal_duration=True  -> video duration split into 7 equal slices, or
  (b) custom per-step durations as a list of 7 numbers (seconds)

A tolerance window (default ±3s) is applied at each step boundary so
windows landing near a transition are not unfairly penalised — they are
counted as correct if they match EITHER the step before or after the
boundary within the tolerance.

Output
------
  - Per-step recall: was each step detected during its expected window?
  - Per-step "missed" flag: steps with near-zero coverage in their slot
  - Timeline plot: predicted (top) vs ground truth (bottom) side by side
  - Overall sequence-following score
"""

import sys, time, json
import numpy as np
import cv2
import torch
import torch.nn.functional as F
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path

try:
    from IPython.display import display, Image as IPImage
    from io import BytesIO
    _IN_NOTEBOOK = True
except ImportError:
    _IN_NOTEBOOK = False

sys.path.insert(0, "/kaggle/working")

# ─────────────────────────────────────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────────────────────────────────────

BASE_DIR  = Path("/kaggle/working/handwash_tcn_dataset_v6")
CKPT_PATH = BASE_DIR / "checkpoints" / "student_best.pt"
NORM_PATH = BASE_DIR / "metadata"    / "norm_stats.npz"
MP_TASK   = Path("/kaggle/working/models/hand_landmarker.task")
OUT_DIR   = BASE_DIR / "sequence_tests"
OUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_FPS  = 15
SEQ_LEN     = 64
STRIDE      = 4
FEATURE_DIM = 403
NUM_CLASSES = 7

MP_DET_CONF  = 0.5
MP_TRK_CONF  = 0.5
MP_PRES_CONF = 0.5
MAX_HANDS    = 2

DEFAULT_TOLERANCE_S = 3.0   # ±3 seconds at boundaries

STEP_NAMES = [
    "Step 1: Rub Palms",  "Step 2: Rub Backs",
    "Step 3: Interlace",  "Step 4: Back Fingers",
    "Step 5: Thumbs",     "Step 6: Fingertips",
    "Step 7: Wrists",
]
STEP_SHORT  = ["S1","S2","S3","S4","S5","S6","S7"]
STEP_COLORS = ["#3B82F6","#10B981","#F59E0B","#EF4444",
               "#8B5CF6","#EC4899","#14B8A6"]

WRIST=0; MIDDLE_MCP=9; INDEX_MCP=5; PINKY_MCP=17
INDEX_TIP=8; MIDDLE_TIP=12; RING_TIP=16; PINKY_TIP=20
HAND_COORDS=63


# ─────────────────────────────────────────────────────────────────────────────
# 1. GROUND TRUTH TIMELINE BUILDER
# ─────────────────────────────────────────────────────────────────────────────

def build_timeline(duration_s, step_durations=None):
    """
    Build a strict sequential ground-truth timeline for 7 WHO steps.

    Parameters
    ----------
    duration_s     : total video duration in seconds
    step_durations : list of 7 numbers (seconds) for each step's duration.
                      If None, splits duration_s into 7 equal slices.
                      If provided but doesn't sum to duration_s, it is
                      rescaled proportionally to fit the actual video length.

    Returns
    -------
    list of (start_s, end_s, step_idx) tuples, step_idx is 0-based
    """
    if step_durations is None:
        slice_len = duration_s / 7
        durations = [slice_len] * 7
    else:
        assert len(step_durations) == 7, "Must provide exactly 7 durations"
        total = sum(step_durations)
        # Rescale to match actual video duration
        durations = [d * duration_s / total for d in step_durations]

    timeline = []
    t = 0.0
    for i, d in enumerate(durations):
        timeline.append((t, t + d, i))
        t += d
    return timeline


def expected_step_at(t, timeline, tolerance=DEFAULT_TOLERANCE_S):
    """
    Return the set of step indices considered "correct" at time t,
    given the tolerance window. Near a boundary, both adjacent steps
    are accepted.
    """
    accepted = set()
    for start, end, step_idx in timeline:
        if (start - tolerance) <= t <= (end + tolerance):
            accepted.add(step_idx)
    return accepted


# ─────────────────────────────────────────────────────────────────────────────
# 2. FEATURE PIPELINE  (v6 — palm-scaled, 403-dim)
# ─────────────────────────────────────────────────────────────────────────────

def _get_lm(frame, hand):
    if hand=="left": return frame[0:63].reshape(21,3), float(frame[126])
    return frame[63:126].reshape(21,3), float(frame[127])

def _palm_scale(lm,p):
    if p<0.5: return 1.0
    s=float(np.linalg.norm(lm[MIDDLE_MCP]-lm[WRIST]))
    return s if s>1e-6 else 1.0

def _palm_normal(lm):
    n=np.cross(lm[INDEX_MCP]-lm[WRIST],lm[PINKY_MCP]-lm[WRIST]).astype(np.float32)
    nm=np.linalg.norm(n); return n/nm if nm>1e-6 else n

def _scale_frame(frame):
    out=frame.copy()
    ll=frame[0:63].reshape(21,3); rl=frame[63:126].reshape(21,3)
    lp=float(frame[126]); rp=float(frame[127])
    out[0:63]=(ll/_palm_scale(ll,lp)).reshape(-1)
    out[63:126]=(rl/_palm_scale(rl,rp)).reshape(-1)
    return out

def _orient(sf):
    ll,lp=_get_lm(sf,"left"); rl,rp=_get_lm(sf,"right")
    ln=_palm_normal(ll) if lp>0.5 else np.zeros(3,np.float32)
    rn=_palm_normal(rl) if rp>0.5 else np.zeros(3,np.float32)
    def zsp(lm,p):
        z=lm[[INDEX_TIP,MIDDLE_TIP,RING_TIP,PINKY_TIP],2]
        return float(z.max()-z.min()) if p>0.5 else 0.
    return np.array([*ln,*rn,float(np.dot(ln,rn)),zsp(ll,lp),zsp(rl,rp)],np.float32)

def _spread(lm,p):
    if p<0.5: return 0.,0.
    t=lm[[INDEX_TIP,MIDDLE_TIP,RING_TIP,PINKY_TIP]]
    d=np.array([np.linalg.norm(t[i]-t[i+1]) for i in range(3)],np.float32)
    return float(d.mean()),float(d.std())

def _cross(own,op,pp,opp):
    if pp<0.5 or opp<0.5: return 0.
    t=own[[INDEX_TIP,MIDDLE_TIP,RING_TIP,PINKY_TIP]]
    return float(np.linalg.norm(t-op[WRIST],axis=1).mean())

def _s6v4(sf):
    ll,lp=_get_lm(sf,"left"); rl,rp=_get_lm(sf,"right")
    lm,ls=_spread(ll,lp); rm,rs=_spread(rl,rp)
    return np.array([lm,ls,rm,rs,_cross(ll,rl,lp,rp),_cross(rl,ll,rp,lp)],np.float32)

def _curl(lm,p):
    if p<0.5: return 0.
    tc=lm[[INDEX_TIP,MIDDLE_TIP,RING_TIP,PINKY_TIP]].mean(0)
    w2t=tc-lm[WRIST]; n=np.linalg.norm(w2t)
    if n<1e-6: return 0.
    pn=_palm_normal(lm); pnn=np.linalg.norm(pn)
    if pnn<1e-6: return 0.
    return float(np.arccos(np.clip(np.dot(w2t/n,pn),-1.,1.)))

def _s6v5(rf):
    ll,lp=_get_lm(rf,"left"); rl,rp=_get_lm(rf,"right")
    return np.array([_curl(ll,lp),_curl(rl,rp)],np.float32)

def mediapipe_to_raw(result):
    left=np.zeros((21,3),np.float32); right=np.zeros((21,3),np.float32)
    lp=rp=0.0
    if result.hand_landmarks:
        for i,lms in enumerate(result.hand_landmarks):
            label=""
            if i<len(result.handedness) and result.handedness[i]:
                label=result.handedness[i][0].category_name
            c=np.array([[lm.x,lm.y,lm.z] for lm in lms],np.float32)
            if label=="Left": left,lp=c,1.0
            else:             right,rp=c,1.0
    return np.concatenate([left.reshape(-1),right.reshape(-1),
                           np.array([lp,rp],np.float32)])

def build_features(base_seq):
    T=base_seq.shape[0]
    sc=np.stack([_scale_frame(base_seq[t]) for t in range(T)])
    wr=sc.copy()
    lw=wr[:,0:3].copy(); rw=wr[:,HAND_COORDS:HAND_COORDS+3].copy()
    for lm in range(21):
        wr[:,lm*3:lm*3+3]-=lw
        wr[:,HAND_COORDS+lm*3:HAND_COORDS+lm*3+3]-=rw
    vel=np.diff(wr,axis=0,prepend=wr[0:1])
    acc=np.diff(vel,axis=0,prepend=vel[0:1])
    kin=np.concatenate([wr,vel,acc],axis=1)
    orient=np.stack([_orient(sc[t]) for t in range(T)])
    s6v4=np.stack([_s6v4(sc[t]) for t in range(T)])
    curl=np.stack([_s6v5(base_seq[t]) for t in range(T)])
    lsv=np.diff(s6v4[:,0],prepend=s6v4[0,0])
    rsv=np.diff(s6v4[:,2],prepend=s6v4[0,2])
    spv=np.stack([lsv,rsv],axis=1)
    return np.concatenate([kin,orient,s6v4,curl,spv],axis=1)


# ─────────────────────────────────────────────────────────────────────────────
# 3. EXTRACTION
# ─────────────────────────────────────────────────────────────────────────────

def extract_base(video_path):
    import mediapipe as mp
    opts=mp.tasks.vision.HandLandmarkerOptions(
        base_options=mp.tasks.BaseOptions(model_asset_path=str(MP_TASK)),
        running_mode=mp.tasks.vision.RunningMode.VIDEO,
        num_hands=MAX_HANDS,
        min_hand_detection_confidence=MP_DET_CONF,
        min_hand_presence_confidence=MP_PRES_CONF,
        min_tracking_confidence=MP_TRK_CONF,
    )
    cap=cv2.VideoCapture(str(video_path))
    orig_fps=cap.get(cv2.CAP_PROP_FPS) or 30.
    total=int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    dur=total/orig_fps
    interval=max(1,int(round(orig_fps/TARGET_FPS)))
    print(f"  Duration   : {dur:.1f}s  ({total} frames @ {orig_fps:.0f}FPS)")
    print(f"  WHO target : 40-60s  →  "
          f"{'✓ within range' if 40<=dur<=60 else '⚠ outside WHO 40-60s range'}")

    feats=[]; fi=0; si=0
    with mp.tasks.vision.HandLandmarker.create_from_options(opts) as lm:
        while True:
            ret,frame=cap.read()
            if not ret: break
            if fi%interval==0:
                rgb=cv2.cvtColor(frame,cv2.COLOR_BGR2RGB)
                img=mp.Image(image_format=mp.ImageFormat.SRGB,data=rgb)
                res=lm.detect_for_video(img,int(si/TARGET_FPS*1000))
                feats.append(mediapipe_to_raw(res)); si+=1
            fi+=1
    cap.release()

    if not feats: return None, dur

    # ── Detailed hand-detection diagnostics ──────────────────────────────────
    left_present  = np.array([f[126] for f in feats])
    right_present = np.array([f[127] for f in feats])
    both_present  = (left_present > 0) & (right_present > 0)
    one_present   = (left_present > 0) ^ (right_present > 0)
    none_present  = (left_present == 0) & (right_present == 0)

    n_hands = int((left_present + right_present > 0).sum())
    print(f"  Extracted  : {len(feats)} frames  |  "
          f"Hands detected: {n_hands} ({100*n_hands/len(feats):.0f}%)")
    print(f"    Both hands  : {both_present.sum():4d} frames ({100*both_present.mean():.0f}%)")
    print(f"    One hand    : {one_present.sum():4d} frames ({100*one_present.mean():.0f}%)")
    print(f"    No hands    : {none_present.sum():4d} frames ({100*none_present.mean():.0f}%)")

    # Per-10-second breakdown — shows WHERE detection drops, not just overall %
    print(f"\n  Hand detection by time segment (10s buckets):")
    n = len(feats)
    seg_len = int(TARGET_FPS * 10)  # frames per 10s bucket
    for seg_start in range(0, n, seg_len):
        seg_end = min(seg_start + seg_len, n)
        t0, t1 = seg_start/TARGET_FPS, seg_end/TARGET_FPS
        seg_both = both_present[seg_start:seg_end].mean() * 100
        seg_one  = one_present[seg_start:seg_end].mean()  * 100
        seg_none = none_present[seg_start:seg_end].mean() * 100
        print(f"    {t0:5.1f}s-{t1:5.1f}s  |  "
              f"both:{seg_both:5.1f}%  one:{seg_one:5.1f}%  none:{seg_none:5.1f}%")

    return np.stack(feats), dur


# ─────────────────────────────────────────────────────────────────────────────
# 4. MODEL
# ─────────────────────────────────────────────────────────────────────────────

def load_model(device):
    from model_tcn_attention import TCNAttentionHAR
    ckpt=torch.load(CKPT_PATH,map_location=device,weights_only=False)
    cfg=ckpt["model_config"]
    model=TCNAttentionHAR(
        input_dim=cfg["input_dim"],num_classes=cfg["num_classes"],
        branch_kernels=tuple(cfg["branch_kernels"]),
        branch_channels=cfg["branch_channels"],
        dilations=tuple(cfg["dilations"]),fusion_dim=cfg["fusion_dim"],
        gru_hidden=cfg["gru_hidden"],dropout=cfg["dropout"],
        use_feature_attn=True,use_layer_norm=False,
    )
    try:    model.load_state_dict(ckpt["model"],strict=True)
    except: model.load_state_dict(ckpt["model"],strict=False)
    model.to(device).eval()
    stats=np.load(NORM_PATH)
    return (model, stats["mean"].astype(np.float32),
            stats["std"].astype(np.float32), ckpt.get("best_val_f1",0))


@torch.no_grad()
def infer(model, feat_seq, mean, std, device):
    T=feat_seq.shape[0]
    norm=((feat_seq-mean)/std).astype(np.float32)
    preds,probs,mids=[],[],[]
    if T < SEQ_LEN:
        pad=np.tile(norm[-1:], (SEQ_LEN-T,1))
        norm=np.concatenate([norm,pad],axis=0); T=SEQ_LEN
    for start in range(0,T-SEQ_LEN+1,STRIDE):
        w=norm[start:start+SEQ_LEN]
        x=torch.from_numpy(w).unsqueeze(0).to(device)
        out=model(x); logits=out[0]
        p=F.softmax(logits,dim=-1).squeeze(0).cpu().numpy()
        preds.append(int(np.argmax(p))); probs.append(p)
        mids.append(start+SEQ_LEN//2)
    return np.array(preds),np.array(probs),np.array(mids)

def smooth(probs,w=3):
    if len(probs)<=1: return probs
    from scipy.ndimage import uniform_filter1d
    return uniform_filter1d(probs,size=min(w,len(probs)),axis=0,mode="nearest")


# ─────────────────────────────────────────────────────────────────────────────
# 5. SCORING — per-step recall against the sequential timeline
# ─────────────────────────────────────────────────────────────────────────────

def score_against_timeline(preds, mids, timeline, tolerance):
    """
    For each prediction window, check if it matches the expected step(s)
    at that timestamp (within tolerance). Returns per-window correctness
    and per-step recall statistics.

    Also returns a "what did the model predict instead" breakdown for
    each GT slot — essential for diagnosing whether a missed step is a
    timeline mismatch or a genuine confusion with a specific other step.
    """
    times = mids / TARGET_FPS
    window_correct  = []
    window_expected = []

    for t, pred in zip(times, preds):
        accepted = expected_step_at(t, timeline, tolerance)
        window_expected.append(sorted(accepted))
        window_correct.append(pred in accepted)

    # Per-step recall: for each GT step's exclusive (non-overlap) time range,
    # did the model predict that step in AT LEAST one window?
    step_detected = [False] * NUM_CLASSES
    step_window_count = [0] * NUM_CLASSES
    step_total_windows = [0] * NUM_CLASSES
    step_confusion = [None] * NUM_CLASSES   # what was predicted instead

    for start, end, step_idx in timeline:
        # Windows whose time falls within [start, end] (core region, no tolerance)
        mask = (times >= start) & (times <= end)
        n_windows_in_slot = int(mask.sum())
        step_total_windows[step_idx] = n_windows_in_slot
        if n_windows_in_slot > 0:
            slot_preds = preds[mask]
            n_correct  = int((slot_preds == step_idx).sum())
            step_window_count[step_idx] = n_correct
            step_detected[step_idx] = n_correct > 0
            # Breakdown of what WAS predicted in this slot
            counts = np.bincount(slot_preds, minlength=NUM_CLASSES)
            step_confusion[step_idx] = counts.tolist()

    return {
        "window_correct":     np.array(window_correct),
        "window_expected":    window_expected,
        "step_detected":      step_detected,
        "step_window_count":  step_window_count,
        "step_total_windows": step_total_windows,
        "step_confusion":     step_confusion,
        "overall_accuracy":   float(np.mean(window_correct)) if len(window_correct)>0 else 0.0,
    }


# ─────────────────────────────────────────────────────────────────────────────
# 6. PLOTTING
# ─────────────────────────────────────────────────────────────────────────────

def plot_sequence_result(preds, probs, mids, timeline, tolerance,
                         video_name, save_path):
    t = mids / TARGET_FPS
    W = len(preds)
    duration = timeline[-1][1]

    fig, axes = plt.subplots(3, 1, figsize=(16, 9),
                              gridspec_kw={"height_ratios":[1,1,2]})
    fig.suptitle(
        f"Sequential WHO Step Test: {video_name}\n"
        f"Duration={duration:.1f}s  ·  Tolerance=±{tolerance:.0f}s  ·  "
        f"WHO target: 40-60s",
        fontsize=12, fontweight="bold",
    )

    # ── Row 0: GROUND TRUTH timeline (expected steps in order) ────────────────
    ax0 = axes[0]
    for start, end, step_idx in timeline:
        ax0.fill_betweenx([0,1], start, end,
                          color=STEP_COLORS[step_idx], alpha=0.9)
        ax0.text((start+end)/2, 0.5, STEP_SHORT[step_idx],
                 ha="center", va="center", fontsize=10,
                 color="white", fontweight="bold")
        # Tolerance shading at boundaries
        ax0.axvspan(max(0,start-tolerance), start+tolerance,
                    color="grey", alpha=0.15, zorder=2)
    ax0.set_xlim(0, duration)
    ax0.set_ylim(0,1); ax0.set_yticks([])
    ax0.set_title("Expected (Ground Truth) — WHO Steps in Strict Order",
                  fontweight="bold")

    # ── Row 1: PREDICTED timeline ──────────────────────────────────────────
    ax1 = axes[1]
    for i in range(W-1):
        ax1.fill_betweenx([0,1], t[i], t[i+1],
                          color=STEP_COLORS[preds[i]], alpha=0.9)
    if W>0:
        ax1.fill_betweenx([0,1], t[-1], duration,
                          color=STEP_COLORS[preds[-1]], alpha=0.9)
    ax1.set_xlim(0, duration)
    ax1.set_ylim(0,1); ax1.set_yticks([])
    ax1.set_title("Predicted Step Timeline", fontweight="bold")
    patches=[mpatches.Patch(color=STEP_COLORS[i],label=STEP_SHORT[i])
             for i in range(NUM_CLASSES)]
    ax1.legend(handles=patches, loc="upper right", ncol=7, fontsize=8)

    # ── Row 2: confidence heatmap with GT boundaries overlaid ────────────────
    ax2 = axes[2]
    im = ax2.imshow(probs.T, aspect="auto", origin="lower", cmap="RdYlGn",
                    vmin=0, vmax=1,
                    extent=[0, duration, -0.5, NUM_CLASSES-0.5],
                    interpolation="nearest")
    for start, end, step_idx in timeline:
        ax2.axvline(start, color="white", lw=1, ls="--", alpha=0.6)
        ax2.axhline(step_idx, color="black", lw=0.3, alpha=0.2)
        ax2.plot([start, end], [step_idx, step_idx],
                 color="black", lw=3, alpha=0.5, solid_capstyle="butt")
    ax2.set_yticks(range(NUM_CLASSES))
    ax2.set_yticklabels(STEP_SHORT, fontsize=10)
    ax2.set_xlabel("Time (seconds)", fontsize=10)
    ax2.set_ylabel("Step", fontsize=10)
    ax2.set_title("Confidence Heatmap (black bar = expected GT step at that time)",
                  fontweight="bold")
    plt.colorbar(im, ax=ax2, label="Probability", pad=0.01)

    plt.tight_layout()
    plt.savefig(save_path, bbox_inches="tight", dpi=130)
    print(f"\n  Plot saved → {save_path}")
    if _IN_NOTEBOOK:
        buf=BytesIO(); fig.savefig(buf,format="png",bbox_inches="tight",dpi=115)
        buf.seek(0); plt.close(fig); display(IPImage(data=buf.read()))
    else:
        plt.close(fig)


# ─────────────────────────────────────────────────────────────────────────────
# 7. MAIN
# ─────────────────────────────────────────────────────────────────────────────

def test_sequence(video_path, step_durations=None,
                  tolerance=DEFAULT_TOLERANCE_S):
    """
    Test the v6 model on a full continuous WHO 7-step handwash video.

    Ground truth assumption: steps occur strictly in order, Step1 -> Step7.

    Parameters
    ----------
    video_path     : path to the video (must show all 7 steps in order)
    step_durations : list of 7 numbers (seconds), or None for equal slices.
                      E.g. [5,5,7,6,6,9,4] for 42s total — will be rescaled
                      to match the actual video duration.
    tolerance       : seconds of buffer at each step boundary (default 3.0)
    """
    VIDEO_EXTS = {".mp4",".avi",".mov",".MP4",".AVI",".MOV",".mkv",".MKV"}
    video_path = Path(video_path)

    # Auto-resolve folders to the video file(s) inside them
    if video_path.is_dir():
        videos = sorted(v for v in video_path.rglob("*")
                        if v.suffix in VIDEO_EXTS and v.is_file())
        if not videos:
            print(f"  ✗ No video files found in folder: {video_path}")
            return None
        if len(videos) > 1:
            print(f"  Found {len(videos)} videos in {video_path}:")
            for i, v in enumerate(videos):
                print(f"    [{i}] {v.name}")
            print(f"  Using [0]: {videos[0].name}")
        video_path = videos[0]
    elif not video_path.exists():
        print(f"  ✗ Video not found: {video_path}")
        return None

    print("="*62)
    print("SEQUENTIAL WHO STEP TEST  (v6 model)")
    print("="*62)
    print(f"  Video     : {video_path.name}")
    print(f"  Tolerance : ±{tolerance}s at boundaries")

    device="cuda" if torch.cuda.is_available() else "cpu"
    print(f"\n[1] Loading model ({device})...")
    model,mean,std,val_f1=load_model(device)
    print(f"  val_f1 = {val_f1:.4f}")

    print(f"\n[2] Extracting MediaPipe landmarks...")
    t0=time.time()
    base,duration=extract_base(video_path)
    print(f"  Done in {time.time()-t0:.1f}s")

    if base is None:
        print("  ✗ Could not extract frames")
        return None

    # Build ground truth timeline
    timeline = build_timeline(duration, step_durations)
    print(f"\n  Ground truth timeline (strict order, Step1→Step7):")
    for start,end,idx in timeline:
        print(f"    {STEP_NAMES[idx]:<22}  {start:6.1f}s — {end:6.1f}s  "
              f"({end-start:.1f}s)")

    print(f"\n[3] Computing v6 features...")
    feat=build_features(base)

    print(f"\n[4] Running inference (stride={STRIDE})...")
    preds,probs,mids=infer(model,feat,mean,std,device)
    probs_s=smooth(probs,w=3)
    preds_s=np.argmax(probs_s,axis=1)

    print(f"\n[5] Scoring against sequential timeline...")
    score = score_against_timeline(preds_s, mids, timeline, tolerance)

    # ── Results ───────────────────────────────────────────────────────────────
    print(f"\n{'='*62}")
    print(f"RESULTS")
    print(f"{'='*62}")
    print(f"\n  Overall window accuracy (within tolerance) : "
          f"{score['overall_accuracy']:.1%}")

    print(f"\n  {'Step':<22}  {'Expected slot':>14}  {'Detected?':>10}  "
          f"{'Hit/Total windows':>18}")
    print("  "+"─"*72)
    missed_steps = []
    for start,end,idx in timeline:
        detected = score["step_detected"][idx]
        hit  = score["step_window_count"][idx]
        tot  = score["step_total_windows"][idx]
        flag = "✓ YES" if detected else "✗ MISSED"
        if not detected: missed_steps.append(idx)
        print(f"  {STEP_NAMES[idx]:<22}  {start:5.1f}-{end:5.1f}s  "
              f"{flag:>10}  {hit:>6d} / {tot:<6d}")

    # ── Confusion breakdown for MISSED steps — what did it predict instead? ──
    if missed_steps:
        print(f"\n  ── What the model predicted instead (missed steps) ──────")
        for idx in missed_steps:
            confusion = score["step_confusion"][idx]
            if confusion is None:
                continue
            print(f"\n  During {STEP_NAMES[idx]} slot, model predicted:")
            ranked = sorted(range(NUM_CLASSES), key=lambda i: -confusion[i])
            tot = sum(confusion)
            for rank_i in ranked:
                if confusion[rank_i] == 0:
                    continue
                pct = 100*confusion[rank_i]/tot if tot>0 else 0
                marker = " ← actually predicted this" if rank_i!=idx else ""
                print(f"      {STEP_NAMES[rank_i]:<22}: "
                      f"{confusion[rank_i]:>3d} windows ({pct:>5.1f}%){marker}")


    print(f"\n  ── Summary ──────────────────────────────────────")
    print(f"     Steps detected : {7-len(missed_steps)}/7")
    if missed_steps:
        print(f"     Steps MISSED   : "
              f"{', '.join(STEP_NAMES[i] for i in missed_steps)}")
        print(f"\n  ⚠  Recommendation: review video for the missed steps —")
        print(f"     either the person did not perform them clearly,")
        print(f"     or the model failed to recognise them.")
    else:
        print(f"     ✓ All 7 WHO steps were detected in their expected time slot")

    # Duration compliance
    print(f"\n  WHO duration compliance:")
    print(f"     Video duration : {duration:.1f}s")
    print(f"     WHO guideline  : 40-60s")
    print(f"     Compliant      : "
          f"{'✓ YES' if 40<=duration<=60 else '✗ NO'}")

    # ── Plot ─────────────────────────────────────────────────────────────────
    stem=video_path.stem[:40]
    save_path=OUT_DIR/f"sequence_{stem}.png"
    print(f"\n[6] Generating plot...")
    plot_sequence_result(preds_s,probs_s,mids,timeline,tolerance,
                         video_path.name,save_path)

    return {
        "video":            str(video_path),
        "duration_s":       duration,
        "who_compliant":    bool(40<=duration<=60),
        "overall_accuracy": score["overall_accuracy"],
        "steps_detected":   7-len(missed_steps),
        "steps_missed":     [STEP_NAMES[i] for i in missed_steps],
        "timeline":         [(s,e,STEP_NAMES[i]) for s,e,i in timeline],
    }


if __name__ == "__main__":
    # Example usage — edit VIDEO_PATH
    result = test_sequence(
        "/kaggle/input/your-dataset/full_handwash.mp4",
        step_durations=None,   # None = equal 1/7 slices
        tolerance=3.0,
    )

In [ ]:
exec(open("/kaggle/working/test_continuous_sequence.py").read())

result = test_sequence(
    "/kaggle/input/<your-recordings>/recording_set_01",
    step_durations=[8, 8, 8, 8, 8, 8, 8],   # 8 seconds per step, totals 56s
    tolerance=3.0,
)

In [ ]:
exec(open("/kaggle/working/test_continuous_sequence.py").read())

result = test_sequence(
    "/kaggle/input/<your-recordings>/recording_set_02",
    step_durations=[8, 8, 8, 8, 8, 8, 8],   # 8 seconds per step, totals 56s
    tolerance=3.0,
)

In [ ]:
exec(open("/kaggle/working/test_continuous_sequence.py").read())

result = test_sequence(
    "/kaggle/input/<your-recordings>/recording_set_03",
    step_durations=[8, 8, 8, 8, 8, 8, 8],   # 8 seconds per step, totals 56s
    tolerance=3.0,
)

In [ ]:
exec(open("/kaggle/working/test_continuous_sequence.py").read())

result = test_sequence(
    "/kaggle/input/<your-recordings>/recording_set_04",
    step_durations=[8, 8, 8, 8, 8, 8, 8],   # 8 seconds per step, totals 56s
    tolerance=3.0,
)

In [ ]:
exec(open("/kaggle/working/test_continuous_sequence.py").read())

result = test_sequence(
    "/kaggle/input/<your-recordings>/recording_set_05",
    step_durations=[8, 8, 8, 8, 8, 8, 8],   # 8 seconds per step, totals 56s
    tolerance=3.0,
)

In [ ]:
exec(open("/kaggle/working/test_continuous_sequence.py").read())

result = test_sequence(
    "/kaggle/input/<your-recordings>/recording_set_06",
    step_durations=[8, 8, 8, 8, 8, 8, 8],   # 8 seconds per step, totals 56s
    tolerance=3.0,
)

In [ ]:
exec(open("/kaggle/working/test_continuous_sequence.py").read())

result = test_sequence(
    "/kaggle/input/<your-recordings>/recording_set_07",
    step_durations=[8, 8, 8, 8, 8, 8, 8],   # 8 seconds per step, totals 56s
    tolerance=3.0,
)

In [ ]:
exec(open("/kaggle/working/test_continuous_sequence.py").read())

result = test_sequence(
    "/kaggle/input/<your-recordings>/recording_set_08",
    step_durations=[8, 8, 8, 8, 8, 8, 8],   # 8 seconds per step, totals 56s
    tolerance=3.0,
)

In [ ]:
exec(open("/kaggle/working/test_continuous_sequence.py").read())

result = test_sequence(
    "/kaggle/input/<your-recordings>/recording_set_09",
    step_durations=[8, 8, 8, 8, 8, 8, 8],   # 8 seconds per step, totals 56s
    tolerance=3.0,
)

In [ ]:
exec(open("/kaggle/working/test_continuous_sequence.py").read())

result = test_sequence(
    "/kaggle/input/<your-recordings>/recording_set_08",
    step_durations=[8, 8, 8, 8, 8, 8, 8],   # 8 seconds per step, totals 56s
    tolerance=3.0,
)

In [ ]:
exec(open("/kaggle/working/test_continuous_sequence.py").read())

result = test_sequence(
    "/kaggle/input/<your-recordings>/recording_set_05",
    step_durations=[8, 8, 8, 8, 8, 8, 8],   # 8 seconds per step, totals 56s
    tolerance=3.0,
)

In [ ]:
exec(open("/kaggle/working/test_continuous_sequence.py").read())

result = test_sequence(
    "/kaggle/input/<your-recordings>/recording_set_07",
    step_durations=[8, 8, 8, 8, 8, 8, 8],   # 8 seconds per step, totals 56s
    tolerance=3.0,
)

In [ ]:
import os
paths = [
    "/kaggle/working/handwash_tcn_dataset_v6/checkpoints/student_best.pt",
    "/kaggle/working/handwash_tcn_dataset_v6/metadata/norm_stats.npz",
    "/kaggle/working/models/hand_landmarker.task",
    "/kaggle/working/model_tcn_attention.py",
]
for p in paths:
    exists = os.path.exists(p)
    size = f"{os.path.getsize(p)/1024/1024:.2f} MB" if exists else "—"
    print(f"{'✓' if exists else '✗'}  {p}  {size}")

In [ ]:
"""
package_for_local_testing.py — Run this ON KAGGLE to bundle everything
==========================================================================
Creates a single zip on Kaggle containing the 4 model files PLUS the
local testing scripts, ready for you to download as ONE file instead
of hunting through Kaggle's file browser for each piece separately.

Run this in a Kaggle code cell, then download the resulting zip from
Kaggle's Output panel (it will appear as
/kaggle/working/local_realtime_testing_bundle.zip).
"""

import os
import shutil
import zipfile
from pathlib import Path

# ── Source paths on Kaggle ───────────────────────────────────────────────────
MODEL_FILES = {
    "/kaggle/working/handwash_tcn_dataset_v6/checkpoints/student_best.pt": "student_best.pt",
    "/kaggle/working/handwash_tcn_dataset_v6/metadata/norm_stats.npz":     "norm_stats.npz",
    "/kaggle/working/models/hand_landmarker.task":                         "hand_landmarker.task",
    "/kaggle/working/model_tcn_attention.py":                              "model_tcn_attention.py",
}

OUTPUT_DIR  = Path("/kaggle/working/local_realtime_bundle")
OUTPUT_ZIP  = Path("/kaggle/working/local_realtime_testing_bundle.zip")
STATIC_DIR  = OUTPUT_DIR / "static"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
STATIC_DIR.mkdir(parents=True, exist_ok=True)

print("="*62)
print("PACKAGING MODEL FILES FOR LOCAL REAL-TIME TESTING")
print("="*62)

missing = []
for src, dst_name in MODEL_FILES.items():
    src_path = Path(src)
    if not src_path.exists():
        print(f"  ✗ MISSING: {src}")
        missing.append(src)
        continue
    dst_path = OUTPUT_DIR / dst_name
    shutil.copy2(src_path, dst_path)
    size_mb = dst_path.stat().st_size / 1024 / 1024
    print(f"  ✓ Copied: {dst_name}  ({size_mb:.2f} MB)")

if missing:
    print(f"\n  ⚠ {len(missing)} file(s) missing — bundle will be incomplete")
else:
    print(f"\n  ✓ All 4 model files copied successfully")

# ── Write a verification README directly into the bundle ────────────────────
readme_content = """# Local Real-Time Testing Bundle

Contains the 4 model files needed to run real-time webcam inference
with the v6 student model (checkpoint: student_best.pt, epoch 30,
val_f1=0.9848, test_f1=0.9731, test_s6_recall=0.85).

Files in this bundle:
  - student_best.pt          (trained model checkpoint)
  - norm_stats.npz           (feature normalisation stats)
  - hand_landmarker.task     (MediaPipe hand detection model)
  - model_tcn_attention.py   (model architecture definition)

NEXT STEPS:
1. Extract this zip on your laptop
2. Download server.py, preflight_check.py, visibility_guard.py,
   webcam_inference_v6.py, and the static/ folder from the earlier
   deployment_bundle.zip (already provided separately)
3. Place ALL files together in one folder:

   pi_webapp/
   |-- server.py
   |-- preflight_check.py
   |-- visibility_guard.py
   |-- webcam_inference_v6.py
   |-- model_tcn_attention.py     <- from this bundle
   |-- student_best.pt             <- from this bundle
   |-- norm_stats.npz              <- from this bundle
   |-- hand_landmarker.task        <- from this bundle
   `-- static/
       |-- index.html
       `-- app.js

4. Run: python preflight_check.py
5. If all checks pass: python server.py
6. Open http://localhost:8000 in your browser
"""

with open(OUTPUT_DIR / "README.txt", "w") as f:
    f.write(readme_content)
print(f"\n  ✓ README.txt written")

# ── Zip it all up ─────────────────────────────────────────────────────────────
if OUTPUT_ZIP.exists():
    OUTPUT_ZIP.unlink()

with zipfile.ZipFile(OUTPUT_ZIP, "w", zipfile.ZIP_DEFLATED) as zf:
    for file in OUTPUT_DIR.rglob("*"):
        if file.is_file():
            arcname = file.relative_to(OUTPUT_DIR)
            zf.write(file, arcname)

zip_size_mb = OUTPUT_ZIP.stat().st_size / 1024 / 1024
print(f"\n{'='*62}")
print(f"✓ BUNDLE CREATED: {OUTPUT_ZIP}")
print(f"  Size: {zip_size_mb:.2f} MB")
print(f"{'='*62}")
print(f"\nNow go to Kaggle's Output/Data panel (right sidebar) and")
print(f"download: local_realtime_testing_bundle.zip")